In [2]:
#Imports
import pandas as pd
import numpy as np
import flask
import plotly.graph_objs as go
import plotly.express as px

import json
import os
import itertools
import random

import numpy as np
import math
from skspatial.objects import Line, Points, Plane
from skspatial.plotting import plot_3d
import json
import pandas as pd
import glob
from tqdm import tqdm

In [4]:
driveDir = "/mnt/d/Lab/CleanGit/TCRSIP/test/"

In [8]:
#@title Dashboard Functions
"""
MolViewer — plain Flask server, fully server-side rendered
===========================================================
No Dash, no WebSockets, no React. Every dropdown change is a normal
GET request; Flask renders the full page and returns HTML. Plotly.js
is loaded from CDN and given the figure JSON directly — no callbacks
needed. Works perfectly through Colab's port proxy.

Usage (Google Colab):
─────────────────────
  Cell 1 — install once:
      !pip install -q flask plotly pandas

  Cell 2 — start & display (cell returns immediately):
      from mol_viewer import MolViewer
      viewer = MolViewer()
      viewer.show()

  Re-display in any later cell:
      viewer.show()

Directory layout expected:
    driveDir/
    ├── TCRtemplate.csv        ← col 0 = structure name, other cols = chain seqs
    └── Structures/
        └── Raw/
            ├── mol_A.pdb
            └── mol_B.pdb
"""

# ── 0. Auto-install ────────────────────────────────────────────────────────────
import subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

for _pkg, _imp in [("flask", "flask"), ("plotly", "plotly"), ("pandas", "pandas")]:
    try:
        __import__(_imp)
    except ImportError:
        _pip(_pkg)

# ── 1. Imports ─────────────────────────────────────────────────────────────────
import os, glob, threading, socket, json, time, logging
import pandas as pd
import plotly.graph_objects as go
from flask import Flask, request, Response

# ── 2. Config ──────────────────────────────────────────────────────────────────
RESULTS_DIR  = os.path.join(f"{driveDir}", "Results")
COLS_PER_ROW = 10
IFRAME_H     = 720

# ── 3. Colour palettes ─────────────────────────────────────────────────────────
CHAIN_COLOURS = [
    "#00d4ff", "#ff6b35", "#7dff6b", "#ff3de8",
    "#ffe135", "#b56bff", "#35ffc8", "#ff3535",
]

AA_BG = {
    **dict.fromkeys(list("AVILMFWP"), "#c8a84b"),
    **dict.fromkeys(list("STCYNQ"),   "#4ba89a"),
    **dict.fromkeys(list("KRH"),      "#4b7ec8"),
    **dict.fromkeys(list("DE"),       "#c84b4b"),
    "G": "#606878",
}

CHAIN_NAMES = {
    'A': 'MHCa',
    'B': 'MHCb',
    'C': 'Peptide',
    'D': 'TCRa',
    'E': 'TCRb',
}

def _chain_label(chain):
    """Return the display name for a PDB chain letter."""
    return CHAIN_NAMES.get(chain, f"Chain {chain}")

def _chain_col(i):
    return CHAIN_COLOURS[i % len(CHAIN_COLOURS)]

# ── 4. PDB parser ──────────────────────────────────────────────────────────────
def parse_pdb(filepath):
    rows = []
    with open(filepath) as fh:
        for line in fh:
            rec = line[:6].strip()
            if rec in ("ATOM", "HETATM") and line[12:16].strip() == "CA":
                try:
                    rows.append({
                        "chain":   line[21].strip() or "A",
                        "resseq":  int(line[22:26]),
                        "resname": line[17:20].strip(),
                        "x":      float(line[30:38]),
                        "y":      float(line[38:46]),
                        "z":      float(line[46:54]),
                        "plddt":  float(line[60:66]),   # column 11 — B-factor / pLDDT
                    })
                except ValueError:
                    pass
    return pd.DataFrame(rows)

# ── 5. Colour scheme helpers ───────────────────────────────────────────────────
COLOR_SCHEMES = ["chain", "plddt"]

PLDDT_DECILE_COLOURS = [
    "#ffdd00",  # 0–10
    "#ffee55",  # 10–20
    "#ffee99",  # 20–30
    "#ffffcc",  # 30–40
    "#ffffff",  # 40–50
    "#cce5ff",  # 50–60
    "#99ccff",  # 60–70
    "#5599ee",  # 70–80
    "#2266cc",  # 80–90
    "#003388",  # 90–100
]

def _plddt_bin_colour(value):
    idx = min(int(float(value) / 10), 9)
    return PLDDT_DECILE_COLOURS[idx]

def _build_colours(sub, scheme, chain_idx):
    """
    Return (pt_colours, ln_colour_arg, opacities) for one chain.
    All colours are plain hex strings — no colorscale needed.
    """
    n = len(sub)
    opacities = [0.9] * n

    if scheme == "plddt" and "plddt" in sub.columns:
        pt_colours   = [_plddt_bin_colour(v) for v in sub["plddt"]]
        # Line: interleave per-segment colours with None gaps
        ln_colour_arg = []
        for k in range(n - 1):
            ln_colour_arg += [pt_colours[k], pt_colours[k+1], "#ffffff"]
    else:
        solid         = _chain_col(chain_idx)
        pt_colours    = [solid] * n
        ln_colour_arg = solid

    return pt_colours, ln_colour_arg, opacities


# ── 5b. Figure builder ─────────────────────────────────────────────────────────
def build_figure_dict(pdb_path, color_scheme="chain",
                      show_points=False, show_lines=True,
                      show_ss=False, struct_json=None):
    """
    Build a Plotly figure.
    Returns (fig_json_str, n_ca, n_res, n_chains, chain_info).
    chain_info: { chain_name: { line_trace: int|None, marker_trace: int|None, resseqs: [int] } }
    Also bakes in one hidden highlight trace per region (opacity=0).
    """
    df = parse_pdb(pdb_path)

    if df.empty:
        fig = go.Figure()
        fig.update_layout(
            paper_bgcolor="#070c1a",
            scene=dict(bgcolor="#080d1c"),
            annotations=[dict(text="No Cα atoms found", x=0.5, y=0.5,
                              xref="paper", yref="paper", showarrow=False,
                              font=dict(color="#3a5070", size=14))],
            margin=dict(l=0, r=0, t=0, b=0),
        )
        return fig.to_json(), 0, 0, 0, {}

    chains     = sorted(df["chain"].unique())
    n_ca       = len(df)
    n_chains   = len(chains)
    n_res      = df["resseq"].nunique()
    traces     = []
    chain_info = {}
    trace_idx  = 0

    # Build per-chain sub-DataFrames first so we can look up coords for regions
    chain_subs = {}
    for i, chain in enumerate(chains):
        sub = df[df["chain"] == chain].sort_values("resseq").reset_index(drop=True)
        chain_subs[_chain_label(chain)] = sub

    for i, chain in enumerate(chains):
        sub      = chain_subs[_chain_label(chain)]
        label    = _chain_label(chain)
        resseqs  = sub["resseq"].tolist()
        pt_colours, ln_colour_arg, opacities = _build_colours(sub, color_scheme, i)
        coords   = list(zip(sub["x"], sub["y"], sub["z"]))

        line_trace_idx   = None
        marker_trace_idx = None

        # ── backbone line ──
        if show_lines:
            xs, ys, zs = [], [], []
            for j in range(len(coords) - 1):
                xs += [coords[j][0], coords[j+1][0], None]
                ys += [coords[j][1], coords[j+1][1], None]
                zs += [coords[j][2], coords[j+1][2], None]

            traces.append(go.Scatter3d(
                x=xs, y=ys, z=zs, mode="lines",
                line=dict(width=5, color=ln_colour_arg),
                name=f"{label} backbone",
                hoverinfo="skip", showlegend=False,
                opacity=1,
            ))
            line_trace_idx = trace_idx
            trace_idx += 1

        # ── Cα markers ──
        if show_points:
            plddt_vals = sub["plddt"].tolist() if "plddt" in sub.columns else None
            if plddt_vals:
                hover = [
                    f"{label} · {rn} {ri}<br>({x:.1f}, {y:.1f}, {z:.1f}) Å<br>pLDDT: {p:.1f}"
                    for ri, rn, x, y, z, p in zip(
                        sub["resseq"], sub["resname"],
                        sub["x"], sub["y"], sub["z"], plddt_vals)
                ]
            else:
                hover = [
                    f"{label} · {rn} {ri}<br>({x:.1f}, {y:.1f}, {z:.1f}) Å"
                    for ri, rn, x, y, z in zip(
                        sub["resseq"], sub["resname"],
                        sub["x"], sub["y"], sub["z"])
                ]

            traces.append(go.Scatter3d(
                x=sub["x"], y=sub["y"], z=sub["z"], mode="markers",
                marker=dict(
                    size=7, color=pt_colours, opacity=1,
                    line=dict(color="#ffffff", width=0.5),
                ),
                name=f"{label} Cα",
                text=hover, hovertemplate="%{text}<extra></extra>",
                showlegend=False, opacity=1,
            ))
            marker_trace_idx = trace_idx
            trace_idx += 1

        chain_info[label] = {
            "line_trace":   line_trace_idx,
            "marker_trace": marker_trace_idx,
            "resseqs":      resseqs,
            "plddt":        sub["plddt"].tolist() if "plddt" in sub.columns else [],
        }

    # ── Secondary structure ribbons + cones ────────────────────────────────────
    if show_ss and struct_json:
        secondary = struct_json.get("Secondary", {})
        for chain in sorted(df["chain"].unique()):
            chain_name = _chain_label(chain)
            chain_ss   = secondary.get(chain_name, {})
            ss_list    = chain_ss.get("Structure", [])
            if not ss_list:
                continue

            sub = df[df["chain"] == chain].sort_values("resseq").reset_index(drop=True)
            coords = list(zip(sub["x"], sub["y"], sub["z"]))
            n      = min(len(coords), len(ss_list))
            chain_colour = _chain_col(list(sorted(df["chain"].unique())).index(chain))

            # Walk residues, collect consecutive same-type (E or H) runs
            i = 0
            while i < n:
                ss = ss_list[i]
                if ss not in ("E", "H"):
                    i += 1
                    continue

                # Find end of this run
                j = i
                while j < n and ss_list[j] == ss:
                    j += 1
                # run is [i, j-1]
                run_coords = coords[i:j]
                colour = chain_colour

                if len(run_coords) >= 2:
                    xs = [c[0] for c in run_coords]
                    ys = [c[1] for c in run_coords]
                    zs = [c[2] for c in run_coords]

                    if ss == "H":
                        # Helix — thick ribbon through all residues
                        traces.append(go.Scatter3d(
                            x=xs, y=ys, z=zs,
                            mode="lines",
                            line=dict(color=colour, width=15),
                            name=f"{chain_name} helix",
                            hoverinfo="skip",
                            showlegend=False,
                            opacity=1,
                        ))
                        trace_idx += 1

                    elif ss == "E":
                        # Sheet ribbon goes all the way to the last residue
                        # so it connects flush with the cone tail
                        traces.append(go.Scatter3d(
                            x=xs, y=ys, z=zs,
                            mode="lines",
                            line=dict(color=colour, width=15),
                            name=f"{chain_name} sheet",
                            hoverinfo="skip",
                            showlegend=False,
                            opacity=1,
                        ))
                        trace_idx += 1

                        # Cone at the last residue, pointing in travel direction
                        last   = run_coords[-1]
                        second = run_coords[-2]
                        ux = last[0] - second[0]
                        uy = last[1] - second[1]
                        uz = last[2] - second[2]
                        traces.append(go.Cone(
                            x=[last[0]], y=[last[1]], z=[last[2]],
                            u=[ux], v=[uy], w=[uz],
                            colorscale=[[0, colour], [1, colour]],
                            showscale=False,
                            sizemode="absolute",
                            sizeref=1.2,
                            anchor="tail",
                            name=f"{chain_name} sheet end",
                            hoverinfo="skip",
                            showlegend=False,
                            opacity=1,
                        ))
                        trace_idx += 1

                i = j

    # ── Hidden region highlight traces ─────────────────────────────────────────
    # One trace per region, opacity=0. JS flips them on/off on hover.
    regions_by_chain = struct_json.get("Region", {}) if struct_json else {}
    # Assign pill colours per chain using _CHAIN_NAME_TO_IDX colours
    _REGION_PILL_COLOURS = [
        "#ff6b35", "#7dff6b", "#ff3de8", "#ffe135",
        "#b56bff", "#35ffc8", "#ff3535", "#00d4ff",
        "#ff9900", "#00ffcc", "#ff00aa", "#aaffaa",
    ]
    region_colour_counter = [0]

    region_trace_map = {}   # region_key → trace_idx

    for chain_name, chain_regions in regions_by_chain.items():
        if not chain_regions:
            continue
        sub = chain_subs.get(chain_name)
        if sub is None:
            continue
        resseq_to_row = {r: row for row, r in enumerate(sub["resseq"].tolist())}

        for region_name, region_data in chain_regions.items():
            if isinstance(region_data, dict):
                start = region_data.get("start", 0)
                end   = region_data.get("end",   0)
            elif isinstance(region_data, (list, tuple)) and len(region_data) >= 2:
                start, end = int(region_data[0]), int(region_data[1])
            else:
                continue

            # Collect xyz for residues in this region
            idxs = [resseq_to_row[r] for r in range(start, end + 1) if r in resseq_to_row]
            if not idxs:
                continue

            rx = sub["x"].iloc[idxs].tolist()
            ry = sub["y"].iloc[idxs].tolist()
            rz = sub["z"].iloc[idxs].tolist()

            # None-separated line segments between consecutive region residues
            lxs, lys, lzs = [], [], []
            for k in range(len(rx) - 1):
                lxs += [rx[k], rx[k+1], None]
                lys += [ry[k], ry[k+1], None]
                lzs += [rz[k], rz[k+1], None]

            colour = _REGION_PILL_COLOURS[
                region_colour_counter[0] % len(_REGION_PILL_COLOURS)
            ]
            region_colour_counter[0] += 1

            # Build hover labels per residue
            hover_pts = [
                f"{chain_name} · {sub['resname'].iloc[i]} {sub['resseq'].iloc[i]}"
                f"<br>({rx[k]:.1f}, {ry[k]:.1f}, {rz[k]:.1f}) Å"
                for k, i in enumerate(idxs)
            ]
            # For line mode: interleave hover per segment (same label for both ends, None for gap)
            hover_lines = []
            for k in range(len(idxs) - 1):
                hover_lines += [hover_pts[k], hover_pts[k+1], None]

            if show_points and show_lines:
                mode = "lines+markers"
                tx, ty, tz   = rx, ry, rz
                hover_text   = hover_pts
            elif show_points:
                mode = "markers"
                tx, ty, tz   = rx, ry, rz
                hover_text   = hover_pts
            else:
                mode = "lines"
                tx, ty, tz   = lxs, lys, lzs
                hover_text   = hover_lines

            region_key = f"{chain_name}__{region_name}"
            traces.append(go.Scatter3d(
                x=tx, y=ty, z=tz,
                mode=mode,
                marker=dict(size=7, color=colour, opacity=1.0,
                            line=dict(color="white", width=0.5)) if show_points else dict(),
                line=dict(color=colour, width=4) if show_lines else dict(),
                text=hover_text,
                hovertemplate="%{text}<extra></extra>",
                name=f"{chain_name} {region_name}",
                showlegend=False,
                opacity=0,   # hidden by default
            ))
            region_trace_map[region_key] = {
                "highlight_trace": trace_idx,
                "colour":          colour,
            }
            trace_idx += 1

    # Store region_trace_map into chain_info for render_page to pass to build_regions_html
    chain_info["__region_trace_map__"] = region_trace_map

    # ── Geometry trace index range (for dimming) ────────────────────────────────
    # Everything before the legend dummies is a geometry trace
    chain_info["__geom_trace_count__"] = trace_idx

    # ── Legend dummy traces ────────────────────────────────────────────────────
    if color_scheme == "chain":
        for i, chain in enumerate(chains):
            traces.append(go.Scatter3d(
                x=[None], y=[None], z=[None], mode="markers",
                marker=dict(size=8, color=_chain_col(i), opacity=1),
                name=_chain_label(chain),
                showlegend=True, hoverinfo="skip",
                legendrank=100 + i,
            ))
            trace_idx += 1
    else:
        decile_labels = [
            "90–100", "80–90", "70–80", "60–70", "50–60",
            "40–50",  "30–40", "20–30", "10–20", "0–10",
        ]
        for rank, (lbl, colour) in enumerate(
            zip(decile_labels, reversed(PLDDT_DECILE_COLOURS))
        ):
            traces.append(go.Scatter3d(
                x=[None], y=[None], z=[None], mode="markers",
                marker=dict(size=8, color=colour, opacity=1),
                name=f"pLDDT {lbl}",
                showlegend=True, hoverinfo="skip",
                legendrank=1000 + rank,
            ))
            trace_idx += 1

    fig = go.Figure(data=traces)
    fig.update_layout(
        paper_bgcolor="#070c1a",
        scene=dict(
            bgcolor="#080d1c",
            xaxis=dict(showbackground=False, gridcolor="#1a2a42",
                       tickfont=dict(color="#3a5070", size=9), title="X (Å)"),
            yaxis=dict(showbackground=False, gridcolor="#1a2a42",
                       tickfont=dict(color="#3a5070", size=9), title="Y (Å)"),
            zaxis=dict(showbackground=False, gridcolor="#1a2a42",
                       tickfont=dict(color="#3a5070", size=9), title="Z (Å)"),
        ),
        legend=dict(font=dict(color="#7aaac8", size=10, family="monospace"),
                    bgcolor="rgba(8,13,28,0.85)", bordercolor="#1a3050", borderwidth=1,
                    yanchor="bottom", y=0.01, xanchor="right", x=0.99),
        scene_camera=dict(
            up=dict(x=0, y=1, z=0),
            eye=dict(x=1.5, y=1.5, z=1.5),
        ),
        margin=dict(l=0, r=0, t=0, b=0),
        uirevision="fixed",
    )
    return fig.to_json(), n_ca, n_res, n_chains, chain_info

    if df.empty:
        fig = go.Figure()
        fig.update_layout(
            paper_bgcolor="#070c1a",
            scene=dict(bgcolor="#080d1c"),
            annotations=[dict(text="No Cα atoms found", x=0.5, y=0.5,
                              xref="paper", yref="paper", showarrow=False,
                              font=dict(color="#3a5070", size=14))],
            margin=dict(l=0, r=0, t=0, b=0),
        )
        return fig.to_json(), 0, 0, 0, {}

    chains     = sorted(df["chain"].unique())
    n_ca       = len(df)
    n_chains   = len(chains)
    n_res      = df["resseq"].nunique()
    traces     = []
    chain_info = {}
    trace_idx  = 0

    for i, chain in enumerate(chains):
        sub      = df[df["chain"] == chain].sort_values("resseq").reset_index(drop=True)
        label    = _chain_label(chain)
        resseqs  = sub["resseq"].tolist()
        pt_colours, ln_colour_arg, opacities = _build_colours(sub, color_scheme, i)
        coords   = list(zip(sub["x"], sub["y"], sub["z"]))

        line_trace_idx   = None
        marker_trace_idx = None

        # ── backbone line ──
        if show_lines:
            xs, ys, zs = [], [], []
            for j in range(len(coords) - 1):
                xs += [coords[j][0], coords[j+1][0], None]
                ys += [coords[j][1], coords[j+1][1], None]
                zs += [coords[j][2], coords[j+1][2], None]

            traces.append(go.Scatter3d(
                x=xs, y=ys, z=zs, mode="lines",
                line=dict(width=5, color=ln_colour_arg),
                name=f"{label} backbone",
                hoverinfo="skip", showlegend=False,
                opacity=1,
            ))
            line_trace_idx = trace_idx
            trace_idx += 1

        # ── Cα markers ──
        if show_points:
            plddt_vals = sub["plddt"].tolist() if "plddt" in sub.columns else None
            if plddt_vals:
                hover = [
                    f"{label} · {rn} {ri}<br>({x:.1f}, {y:.1f}, {z:.1f}) Å<br>pLDDT: {p:.1f}"
                    for ri, rn, x, y, z, p in zip(
                        sub["resseq"], sub["resname"],
                        sub["x"], sub["y"], sub["z"], plddt_vals)
                ]
            else:
                hover = [
                    f"{label} · {rn} {ri}<br>({x:.1f}, {y:.1f}, {z:.1f}) Å"
                    for ri, rn, x, y, z in zip(
                        sub["resseq"], sub["resname"],
                        sub["x"], sub["y"], sub["z"])
                ]

            traces.append(go.Scatter3d(
                x=sub["x"], y=sub["y"], z=sub["z"], mode="markers",
                marker=dict(
                    size=7,
                    color=pt_colours,
                    opacity=1,
                    line=dict(color="#ffffff", width=0.5),
                ),
                name=f"{label} Cα",
                text=hover, hovertemplate="%{text}<extra></extra>",
                showlegend=False,
                opacity=1,
            ))
            marker_trace_idx = trace_idx
            trace_idx += 1

        chain_info[_chain_label(chain)] = {
            "line_trace":   line_trace_idx,
            "marker_trace": marker_trace_idx,
            "resseqs":      resseqs,
            "plddt":        sub["plddt"].tolist() if "plddt" in sub.columns else [],
        }

    # ── Legend dummy traces ────────────────────────────────────────────────────
    if color_scheme == "chain":
        for i, chain in enumerate(chains):
            traces.append(go.Scatter3d(
                x=[None], y=[None], z=[None], mode="markers",
                marker=dict(size=8, color=_chain_col(i), opacity=1),
                name=_chain_label(chain),
                showlegend=True, hoverinfo="skip",
                legendrank=100 + i,
            ))
            trace_idx += 1
    else:
        decile_labels = [
            "90–100", "80–90", "70–80", "60–70", "50–60",
            "40–50",  "30–40", "20–30", "10–20", "0–10",
        ]
        for rank, (lbl, colour) in enumerate(
            zip(decile_labels, reversed(PLDDT_DECILE_COLOURS))
        ):
            traces.append(go.Scatter3d(
                x=[None], y=[None], z=[None], mode="markers",
                marker=dict(size=8, color=colour, opacity=1),
                name=f"pLDDT {lbl}",
                showlegend=True, hoverinfo="skip",
                legendrank=1000 + rank,
            ))
            trace_idx += 1

    fig = go.Figure(data=traces)
    fig.update_layout(
        paper_bgcolor="#070c1a",
        scene=dict(
            bgcolor="#080d1c",
            xaxis=dict(showbackground=False, gridcolor="#1a2a42",
                       tickfont=dict(color="#3a5070", size=9), title="X (Å)"),
            yaxis=dict(showbackground=False, gridcolor="#1a2a42",
                       tickfont=dict(color="#3a5070", size=9), title="Y (Å)"),
            zaxis=dict(showbackground=False, gridcolor="#1a2a42",
                       tickfont=dict(color="#3a5070", size=9), title="Z (Å)"),
        ),
        legend=dict(font=dict(color="#7aaac8", size=10, family="monospace"),
                    bgcolor="rgba(8,13,28,0.85)", bordercolor="#1a3050", borderwidth=1,
                    yanchor="bottom", y=0.01, xanchor="right", x=0.99),
        scene_camera=dict(
            up=dict(x=0, y=1, z=0),
            eye=dict(x=1.5, y=1.5, z=1.5),
        ),
        margin=dict(l=0, r=0, t=0, b=0),
        uirevision="fixed",
    )
    return fig.to_json(), n_ca, n_res, n_chains, chain_info

# ── 6. CSV helpers ─────────────────────────────────────────────────────────────
_csv_cache = None

def load_csv():
    global _csv_cache
    if _csv_cache is not None:
        return _csv_cache
    if not os.path.exists(CSV_PATH):
        _csv_cache = pd.DataFrame()
    else:
        try:
            _csv_cache = pd.read_csv(CSV_PATH)
        except Exception:
            _csv_cache = pd.DataFrame()
    return _csv_cache


def get_molecule_row(df, stem):
    if df.empty or not stem:
        return None
    col0 = df.columns[0]
    exact = df[df[col0].astype(str).str.strip() == stem]
    if not exact.empty:
        return exact.iloc[0]
    partial = df[df[col0].astype(str).str.contains(stem, case=False, na=False)]
    if not partial.empty:
        return partial.iloc[0]
    return None

# ── 6b. JSON loader ────────────────────────────────────────────────────────────
def load_structure_json(stem):
    """Load driveDir/Results/<stem>/<stem>.json. Returns dict or minimal placeholder."""
    path = os.path.join(RESULTS_DIR, stem, f"{stem}.json")
    if not os.path.exists(path):
        return None
    try:
        with open(path) as fh:
            return json.load(fh)
    except Exception:
        return None


def _placeholder_panel(label="No information available"):
    return (
        f'<div style="display:flex;align-items:center;justify-content:center;'
        f'height:100%;color:#3a5070;font-style:italic;font-size:13px;">'
        f'{label}</div>'
    )

# ── 7. Sequence grid HTML builder ──────────────────────────────────────────────
def _aa_tile(letter, pos, bg):
    return (
        f'<div title="Pos {pos}: {letter}" style="width:22px;height:34px;'
        f'border-radius:3px;background:{bg};display:inline-flex;flex-direction:column;'
        f'align-items:center;justify-content:center;flex-shrink:0;margin:1px;">'
        f'<span style="font-size:6px;color:rgba(0,0,0,0.5);line-height:1;">{pos}</span>'
        f'<span style="font-size:11px;font-weight:700;color:rgba(0,0,0,0.85);'
        f'line-height:1.1;">{letter}</span></div>'
    )

_AA_LEGEND = [
    ("#c8a84b", "Hydrophobic",  "A V I L M F W P"),
    ("#4ba89a", "Polar",        "S T C Y N Q"),
    ("#4b7ec8", "Positive",     "K R H"),
    ("#c84b4b", "Negative",     "D E"),
    ("#606878", "Glycine",      "G"),
]

def _build_aa_legend_html():
    items = "".join(
        f'<div style="display:flex;align-items:center;gap:5px;margin-bottom:4px;">'
        f'<div style="width:12px;height:12px;border-radius:2px;background:{bg};flex-shrink:0;"></div>'
        f'<span style="font-size:9px;color:#4a6080;white-space:nowrap;">'
        f'<span style="color:#7aaac8;">{label}</span> &nbsp;{aas}</span>'
        f'</div>'
        for bg, label, aas in _AA_LEGEND
    )
    pdb_items = (
        f'<div style="display:flex;align-items:center;gap:5px;margin-bottom:4px;">'
        f'<div style="width:12px;height:12px;border-radius:2px;background:#1a3a5c;flex-shrink:0;"></div>'
        f'<span style="font-size:9px;color:#4a6080;">Match</span>'
        f'</div>'
        f'<div style="display:flex;align-items:center;gap:5px;margin-bottom:4px;">'
        f'<div style="width:12px;height:12px;border-radius:2px;background:#3d0a0a;flex-shrink:0;"></div>'
        f'<span style="font-size:9px;color:#4a6080;">Mismatch</span>'
        f'</div>'
        f'<div style="display:flex;align-items:center;gap:5px;margin-bottom:4px;">'
        f'<div style="width:12px;height:12px;border-radius:2px;background:#0d0d0d;border:1px solid #1a2a42;flex-shrink:0;"></div>'
        f'<span style="font-size:9px;color:#4a6080;">No data</span>'
        f'</div>'
    )
    return (
        f'<div style="border-top:1px solid #1a2a42;margin-top:10px;padding:8px 10px 4px;'
        f'display:flex;gap:24px;flex-wrap:wrap;">'
        f'<div>'
        f'<div style="font-size:8px;letter-spacing:0.15em;text-transform:uppercase;'
        f'color:#3a5070;margin-bottom:6px;">Sequence Residues</div>'
        f'{items}'
        f'</div>'
        f'<div>'
        f'<div style="font-size:8px;letter-spacing:0.15em;text-transform:uppercase;'
        f'color:#3a5070;margin-bottom:6px;">PDB Residues</div>'
        f'{pdb_items}'
        f'</div>'
        f'</div>'
    )


# Turn → colour mapping
_TURN_COLOURS = {
    "A":    "#5b8dd9", "B":    "#6ab187", "C":    "#e8a838",
    "C+":   "#d4702a", "C++":  "#b84c1e", "D":    "#9b59b6",
    "E":    "#1abc9c", "F":    "#e74c3c", "G":    "#3498db",
    "cdr1": "#ff4d6d", "cdr2": "#ff9f1c", "cdr3": "#c77dff",
    "none": "#2a3a50",
}

def _turn_colour(turn):
    return _TURN_COLOURS.get(turn, "#2a3a50")


def build_consensus_html(struct_json):
    if struct_json is None:
        return '<p style="color:#3a5070;font-style:italic;padding:12px;font-family:monospace;">No JSON file found.</p>'

    sequences   = struct_json.get("Sequence", {})
    chain_order = [c for c in _STRUCT_CHAIN_ORDER if c in sequences]

    cards = []
    for ci, chain_name in enumerate(chain_order):
        val     = sequences[chain_name]
        if not isinstance(val, dict):
            continue
        cons = val.get("Consensus")
        seq  = val.get("Seq", "").strip()
        if not cons or not seq:
            continue

        imgt_nums = cons.get("IMGT", [])
        turns     = cons.get("Turn", [])
        confidence = cons.get("Confidence", None)
        cons_score = val.get("Consensusscore", None)
        accent    = _chain_col(_CHAIN_NAME_TO_IDX.get(chain_name, ci))
        body_id   = f"cons-body-{chain_name.lower()}"

        # Clamp to shortest length
        n = min(len(seq), len(imgt_nums), len(turns))

        # Build paired tiles: IMGT circle on top, AA tile below
        tiles_html = ""
        for k in range(n):
            aa       = seq[k]
            imgt_n   = imgt_nums[k]
            turn     = turns[k]
            t_col    = _turn_colour(turn)
            aa_bg    = "#5b9bd5"   # bland light blue for residues

            imgt_label = str(imgt_n) if imgt_n != -1 else "·"

            # IMGT circle
            circle = (
                f'<div title="{turn}" style="width:28px;height:28px;border-radius:50%;'
                f'background:{t_col};display:flex;align-items:center;justify-content:center;'
                f'flex-shrink:0;">'
                f'<span style="font-size:9px;font-weight:700;color:rgba(255,255,255,0.95);'
                f'line-height:1;">{imgt_label}</span>'
                f'</div>'
            )

            # AA tile (same size as sequence tab)
            aa_tile = (
                f'<div title="Pos {k+1}: {aa} | IMGT {imgt_label} | {turn}" '
                f'style="width:28px;height:28px;border-radius:3px;'
                f'background:{aa_bg};display:flex;align-items:center;justify-content:center;'
                f'flex-shrink:0;">'
                f'<span style="font-size:13px;font-weight:700;color:rgba(0,0,0,0.8);'
                f'line-height:1;">{aa}</span>'
                f'</div>'
            )

            tiles_html += (
                f'<div style="display:flex;flex-direction:column;align-items:center;'
                f'gap:2px;flex-shrink:0;">'
                f'{circle}{aa_tile}'
                f'</div>'
            )

        body_html = (
            f'<div style="display:flex;flex-wrap:nowrap;overflow-x:auto;'
            f'gap:2px;padding-bottom:4px;">'
            f'<div style="display:flex;flex-direction:column;gap:2px;flex-shrink:0;'
            f'justify-content:space-around;padding-right:4px;border-right:1px solid #1a2a42;'
            f'margin-right:4px;">'
            f'<span style="font-size:9px;color:#4a6080;white-space:nowrap;line-height:28px;">IMGT #</span>'
            f'<span style="font-size:9px;color:#4a6080;white-space:nowrap;line-height:28px;">Sequence</span>'
            f'</div>'
            f'{tiles_html}</div>'
        )

        # Count residues with valid IMGT number (>0)
        n_consensus = sum(1 for v in imgt_nums[:n] if v > 0)

        # Legend for Turn colours
        legend_items = "".join(
            f'<div style="display:flex;align-items:center;gap:4px;margin-right:10px;margin-bottom:3px;">'
            f'<div style="width:11px;height:11px;border-radius:50%;background:{col};flex-shrink:0;"></div>'
            f'<span style="font-size:10px;color:#7aaac8;">{name}</span>'
            f'</div>'
            for name, col in _TURN_COLOURS.items() if name != "none"
        )
        legend_html = (
            f'<div style="display:flex;flex-wrap:wrap;padding:6px 10px 4px;'
            f'border-top:1px solid #1a2a42;">{legend_items}</div>'
        )

        score_lbl = ""
        if cons_score is not None:
            s_col = "#4bc87a" if cons_score == 0 else "#ffe135" if cons_score <= 2 else "#c84b4b"
            score_lbl = (
                f'<span style="font-size:9px;color:#4a6080;">Score:</span>'
                f'<span style="font-size:12px;font-weight:700;color:{s_col};margin-left:4px;">{cons_score}</span>'
            )
        conf_lbl = (
            f'<span style="font-size:9px;color:#4a6080;margin-left:8px;">conf</span>'
            f'<span style="font-size:11px;font-weight:700;color:#00d4ff;margin-left:4px;">'
            f'{confidence:.2f}</span>'
        ) if confidence is not None else ""

        cards.append(f"""
<div style="border:1px solid #1a2a42;border-radius:6px;margin-bottom:8px;background:#0a1020;overflow:hidden;">
  <div style="display:flex;align-items:center;justify-content:space-between;
              padding:5px 10px;background:rgba(255,255,255,0.03);border-bottom:1px solid #1a2a42;
              cursor:pointer;user-select:none;"
       onclick="toggleSection('{body_id}', this)">
    <div style="display:flex;align-items:center;">
      <span style="color:{accent};margin-right:6px;font-size:14px;">&#9679;</span>
      <span style="font-weight:700;color:#c8ddf0;font-size:11px;letter-spacing:0.06em;font-family:monospace;">{chain_name}</span>
      <span style="color:#3a5070;font-size:9px;margin-left:8px;">{n_consensus} consensus residues found</span>
      {conf_lbl}
    </div>
    <div style="display:flex;align-items:center;gap:8px;">
      {score_lbl}
      <span class="collapse-arrow" style="color:#4a6080;font-size:11px;transition:transform 0.2s;">&#9650;</span>
    </div>
  </div>
  <div id="{body_id}">
    <div style="padding:6px 10px;overflow-x:auto;">{body_html}</div>
    {legend_html}
  </div>
</div>""")

    if not cards:
        return '<p style="color:#3a5070;font-style:italic;padding:12px;">No consensus data available.</p>'

    return "\n".join(cards)


def build_sequence_html(struct_json):
    if struct_json is None:
        return '<p style="color:#3a5070;font-style:italic;padding:12px;font-family:monospace;">No JSON file found for this structure.</p>'

    sequences = struct_json.get("Sequence", {})
    if not sequences:
        return '<p style="color:#3a5070;font-family:monospace;padding:12px;">No sequence data found.</p>'

    chain_order = [c for c in _STRUCT_CHAIN_ORDER if c in sequences]

    cards = []
    for ci, chain_name in enumerate(chain_order):
        val      = sequences[chain_name]
        seq      = (val.get("Seq",    "") if isinstance(val, dict) else str(val or "")).strip()
        pdbseq   = (val.get("PDBseq", "") if isinstance(val, dict) else "").strip()
        exact    = val.get("Exact",   []) if isinstance(val, dict) else []
        seqscore = val.get("seqscore", None) if isinstance(val, dict) else None
        accent   = _chain_col(_CHAIN_NAME_TO_IDX.get(chain_name, ci))
        is_empty = not seq or seq.lower() in ("nan", "none", "")
        body_id  = f"seq-body-{chain_name.lower()}"

        if is_empty:
            body_html = '<p style="color:#3a5070;font-style:italic;font-size:11px;padding:4px 0;">— no sequence —</p>'
        else:
            # ── Seq tiles (top row) ───────────────────────────────────────────
            def _tile_bg(aa, k):
                try:
                    pdb_aa = pdbseq[k]
                    if pdb_aa != aa:
                        return "#c84b4b"
                except Exception:
                    pass
                return AA_BG.get(aa.upper(), "#2a3a50")

            seq_tiles = "".join(
                _aa_tile(aa, k + 1, _tile_bg(aa, k))
                for k, aa in enumerate(seq)
            )

            # ── PDBseq tiles (bottom row) ─────────────────────────────────────
            MATCH_COL    = "#1a3a5c"   # dark blue — match
            MISMATCH_COL = "#3d0a0a"   # dark red — no match
            NO_DATA_COL  = "#0d0d0d"   # near-black — beyond Exact length

            pdb_tiles = ""
            for k, pdb_aa in enumerate(pdbseq):
                try:
                    is_mismatch = exact[k]
                    bg = MISMATCH_COL if is_mismatch else MATCH_COL
                except IndexError:
                    bg = NO_DATA_COL
                pdb_tiles += (
                    f'<div title="Pos {k+1}: {pdb_aa}" '
                    f'style="width:22px;height:34px;border-radius:3px;background:{bg};'
                    f'display:inline-flex;flex-direction:column;align-items:center;'
                    f'justify-content:center;flex-shrink:0;margin:1px;">'
                    f'<span style="font-size:6px;color:rgba(255,255,255,0.3);line-height:1;">{k+1}</span>'
                    f'<span style="font-size:11px;font-weight:700;color:rgba(255,255,255,0.5);'
                    f'line-height:1.1;">{pdb_aa}</span>'
                    f'</div>'
                )

            # Row label column — "Sequence" and "PDB residues" aligned to rows
            label_col = (
                f'<div style="display:flex;flex-direction:column;gap:2px;flex-shrink:0;'
                f'padding-right:6px;border-right:1px solid #1a2a42;margin-right:4px;">'
                f'<div style="height:34px;display:flex;align-items:center;">'
                f'<span style="font-size:8px;color:#4a6080;white-space:nowrap;">Sequence</span>'
                f'</div>'
                f'<div style="height:34px;display:flex;align-items:center;">'
                f'<span style="font-size:8px;color:#4a6080;white-space:nowrap;">PDB residues</span>'
                f'</div>'
                f'</div>'
            )

            seq_row = f'<div style="display:flex;flex-wrap:nowrap;gap:2px;">{seq_tiles}</div>'
            pdb_row = f'<div style="display:flex;flex-wrap:nowrap;gap:2px;">{pdb_tiles}</div>'

            body_html = (
                f'<div style="overflow-x:auto;overflow-y:hidden;'
                f'display:flex;align-items:center;">'
                f'{label_col}'
                f'<div style="display:flex;flex-direction:column;gap:2px;flex-shrink:0;">'
                f'{seq_row}{pdb_row}'
                f'</div>'
                f'</div>'
            )

        length_lbl = f'<span style="color:#3a5070;font-size:10px;margin-left:8px;">{len(seq)} aa</span>' if not is_empty else ""

        if seqscore is not None:
            score_col = "#4bc87a" if seqscore == 0 else "#ffe135" if seqscore <= 2 else "#c84b4b"
            score_lbl = (
                f'<span style="font-size:9px;color:#4a6080;margin-right:4px;">Score (lower is better):</span>'
                f'<span style="font-size:12px;font-weight:700;color:{score_col};">{seqscore}</span>'
            )
        else:
            score_lbl = ""

        cards.append(f"""
<div style="border:1px solid #1a2a42;border-radius:6px;margin-bottom:8px;background:#0a1020;">
  <div style="display:flex;align-items:center;justify-content:space-between;
              padding:5px 10px;background:rgba(255,255,255,0.03);border-bottom:1px solid #1a2a42;">
    <div style="display:flex;align-items:center;">
      <span style="color:{accent};margin-right:6px;font-size:14px;">&#9679;</span>
      <span style="font-weight:700;color:#c8ddf0;font-size:11px;letter-spacing:0.06em;font-family:monospace;">{chain_name}</span>
      {length_lbl}
    </div>
    <div style="display:flex;align-items:center;gap:8px;">
      {score_lbl}
    </div>
  </div>
  <div style="padding:6px 10px;">{body_html}</div>
  <div style="height:4px;background:rgba(255,255,255,0.03);border-top:1px solid #1a2a42;"></div>
</div>""")

    return "\n".join(cards) + _build_aa_legend_html()


# ── 8a. Structure tab builder ──────────────────────────────────────────────────
# Chain display order for the Structure tab
_STRUCT_CHAIN_ORDER = ["MHCa", "MHCb", "Peptide", "TCRa", "TCRb"]

# Map chain name → colour index (matches CHAIN_NAMES order A-E)
_CHAIN_NAME_TO_IDX = {v: i for i, v in enumerate(CHAIN_NAMES.values())}

def _sse_plot_html(annotation, sse, chain_name, region=None):
    """Call viz_sec() and embed the resulting Plotly figure inline."""
    try:
        sse_fig = viz_sec(annotation, sse, region)
        if sse_fig is None:
            raise ValueError("viz_sec returned None")

        sse_fig.update_layout(
            margin=dict(l=4, r=50, t=4, b=4),
            height=110,
        )

        plot_id  = f"sse-{chain_name.lower()}"
        fig_json = sse_fig.to_json()
        return (
            f'<div id="{plot_id}" style="width:100%;height:110px;display:block;"></div>'
            f'<script>(function(){{'
            f'  var d={fig_json};'
            f'  Plotly.newPlot("{plot_id}",d.data,d.layout,{{responsive:true,displayModeBar:false}});'
            f'}})();</script>'
        )
    except NameError as e:
        return (
            f'<div style="height:110px;display:flex;align-items:center;justify-content:center;'
            f'color:#c84b4b;font-size:10px;font-style:italic;">'
            f'Function not in scope: {e}</div>'
        )
    except Exception as e:
        return (
            f'<div style="height:110px;display:flex;align-items:center;justify-content:center;'
            f'color:#c84b4b;font-size:10px;font-style:italic;">'
            f'{type(e).__name__}: {e}</div>'
        )


def build_regions_html(struct_json, chain_info):
    """
    Build the Regions section HTML using region_trace_map embedded in chain_info.
    Returns (html_str, trace_map_json_str).
    trace_map: { region_key: { highlight_trace: int, geom_trace_count: int, colour: str } }
    """
    if struct_json is None:
        return "", "{}"

    regions_by_chain = struct_json.get("Region", {})
    sequences        = struct_json.get("Sequence", {})
    region_trace_map = chain_info.get("__region_trace_map__", {})
    geom_trace_count = chain_info.get("__geom_trace_count__", 0)

    def _get_seq(chain_name):
        """Extract the true sequence from the new nested Sequence structure."""
        val = sequences.get(chain_name, "")
        if isinstance(val, dict):
            return val.get("Seq", "")
        return val   # fallback: old flat string format

    if not regions_by_chain or not region_trace_map:
        return "", "{}"

    trace_map    = {}
    chain_cards  = {}   # chain_name → list of pill html strings

    for chain_name, chain_regions in regions_by_chain.items():
        if not chain_regions:
            continue
        seq = _get_seq(chain_name)

        for region_name, region_data in chain_regions.items():
            if isinstance(region_data, dict):
                start   = region_data.get("start", 0)
                end     = region_data.get("end",   0)
                reg_seq = region_data.get("sequence", seq[start-1:end] if seq else "")
            elif isinstance(region_data, (list, tuple)) and len(region_data) >= 2:
                start, end = int(region_data[0]), int(region_data[1])
                reg_seq    = seq[start-1:end] if seq else ""
            else:
                continue

            region_key = f"{chain_name}__{region_name}"
            rtm        = region_trace_map.get(region_key, {})
            if not rtm:
                continue

            colour     = rtm["colour"]
            hl_trace   = rtm["highlight_trace"]
            length     = end - start + 1

            trace_map[region_key] = {
                "highlight_trace":  hl_trace,
                "geom_trace_count": geom_trace_count,
                "colour":           colour,
            }

            tooltip = f"{region_name} | {chain_name} | {length} aa | {start}–{end}"
            pill = (
                f'<div class="region-pill" '
                f'data-key="{region_key}" '
                f'data-tip="{tooltip}" '
                f'data-seq="{reg_seq}" '
                f'style="display:flex;align-items:center;gap:5px;'
                f'padding:4px 8px;margin-bottom:3px;border-radius:4px;cursor:pointer;'
                f'border:1px solid {colour};background:#0a1020;'
                f'font-family:monospace;font-size:10px;color:#c8ddf0;'
                f'white-space:nowrap;transition:background 0.15s;">'
                f'<span style="color:{colour};font-size:11px;">&#9679;</span>'
                f'<span>{region_name}</span>'
                f'<span style="color:#3a5070;font-size:9px;">{start}–{end}</span>'
                f'</div>'
            )
            chain_cards.setdefault(chain_name, []).append(pill)

    if not chain_cards:
        return "", "{}"

    # Build one card per chain, displayed horizontally
    cards_html = ""
    for chain_name, pills in chain_cards.items():
        chain_col = _chain_col(_CHAIN_NAME_TO_IDX.get(chain_name, 0))
        pills_html = "\n".join(pills)
        cards_html += f"""
<div style="flex-shrink:0;background:#0a1020;border:1px solid #1a2a42;
            border-radius:6px;overflow:hidden;min-width:120px;">
  <div style="display:flex;align-items:center;gap:5px;padding:5px 8px;
              background:rgba(255,255,255,0.03);border-bottom:1px solid #1a2a42;">
    <span style="color:{chain_col};font-size:11px;">&#9679;</span>
    <span style="font-weight:700;color:#c8ddf0;font-size:10px;
                 font-family:monospace;letter-spacing:0.06em;">{chain_name}</span>
  </div>
  <div style="padding:6px 6px;display:flex;flex-direction:column;align-items:flex-start;">
    {pills_html}
  </div>
</div>"""

    html = f"""
<div style="flex-shrink:0;border-bottom:1px solid #1a2a42;background:#080e1e;">
  <div style="display:flex;align-items:center;justify-content:space-between;
              padding:6px 10px;cursor:pointer;user-select:none;"
       onclick="toggleSection('regions-body', this)">
    <span style="font-size:9px;letter-spacing:0.16em;text-transform:uppercase;
                 color:#4a6080;">Regions</span>
    <div style="display:flex;align-items:center;gap:8px;">
      <span style="font-size:9px;color:#3a5070;font-style:italic;">Hover to highlight</span>
      <span class="collapse-arrow" style="color:#4a6080;font-size:11px;transition:transform 0.2s;">&#9650;</span>
    </div>
  </div>
  <div id="regions-body" style="padding:6px 8px 8px;display:flex;flex-direction:row;
              flex-wrap:wrap;gap:6px;overflow-x:auto;" id="region-pills">
    {cards_html}
  </div>
  <div id="region-tooltip" style="display:none;position:fixed;z-index:9999;
       background:#0d1528;border:1px solid #1b2d4f;border-radius:6px;
       padding:8px 12px;font-family:monospace;font-size:10px;
       color:#c8ddf0;pointer-events:none;max-width:320px;
       box-shadow:0 4px 16px rgba(0,0,0,0.5);">
  </div>
</div>"""

    return html, json.dumps(trace_map)


def build_plddt_html(chain_info, struct_json=None):
    """Build the pLDDT sub-tab: one Plotly line plot per chain with region annotations."""
    if not chain_info:
        return '<div style="display:flex;align-items:center;justify-content:center;height:100%;color:#3a5070;font-style:italic;font-size:13px;">No pLDDT data available.</div>'

    # Regions that go on the higher y tier (further above the plot)
    HIGH_TIER = {"TCRca", "TCRcb", "TCRva", "TCRvb"}
    Y_LOW  = 108
    Y_HIGH = 122
    Y_MAX  = 135

    # Region colours — reuse the pill colours from build_figure_dict
    _REGION_PILL_COLOURS = [
        "#ff6b35", "#7dff6b", "#ff3de8", "#ffe135",
        "#b56bff", "#35ffc8", "#ff3535", "#00d4ff",
        "#ff9900", "#00ffcc", "#ff00aa", "#aaffaa",
    ]

    # Pre-read all regions from struct_json for annotation
    regions_by_chain  = struct_json.get("Region", {}) if struct_json else {}
    plddt_region_data = struct_json.get("pLDDT",  {}) if struct_json else {}

    cards = []
    region_colour_counter = 0

    for chain_name in _STRUCT_CHAIN_ORDER:
        info       = chain_info.get(chain_name, {})
        plddt_vals = info.get("plddt", [])
        if not plddt_vals:
            continue

        chain_colour  = _chain_col(_CHAIN_NAME_TO_IDX.get(chain_name, 0))
        x_vals        = list(range(1, len(plddt_vals) + 1))
        avg           = sum(plddt_vals) / len(plddt_vals)
        avg_col       = "#4bc87a" if avg >= 70 else "#ffe135" if avg >= 50 else "#c84b4b"
        chain_regions = regions_by_chain.get(chain_name, {})
        chain_plddt_regions = plddt_region_data.get(chain_name, {})

        # Build region shapes, annotations, and scores sidebar
        shapes             = []
        annotations        = []
        region_scores_html = ""

        for region_name, region_data in chain_regions.items():
            if isinstance(region_data, dict):
                start = region_data.get("start", 0)
                end   = region_data.get("end",   0)
            elif isinstance(region_data, (list, tuple)) and len(region_data) >= 2:
                start, end = int(region_data[0]), int(region_data[1])
            else:
                continue

            colour = _REGION_PILL_COLOURS[region_colour_counter % len(_REGION_PILL_COLOURS)]
            region_colour_counter += 1

            y_pos   = Y_HIGH if region_name in HIGH_TIER else Y_LOW
            mid_x   = (start + end) / 2

            # Horizontal line spanning the region
            shapes.append(dict(
                type="line",
                x0=start, x1=end,
                y0=y_pos, y1=y_pos,
                line=dict(color=colour, width=3),
                xref="x", yref="y",
            ))
            # Text label above the line for both tiers
            annotations.append(dict(
                x=mid_x, y=y_pos,
                xref="x", yref="y",
                text=region_name,
                showarrow=False,
                font=dict(size=9, color=colour),
                yanchor="bottom",
                xanchor="center",
                yshift=2,
            ))

            # Region average pLDDT score row
            region_avg = chain_plddt_regions.get(region_name)
            if region_avg is not None:
                try:
                    r_avg     = float(region_avg)
                    r_col     = "#4bc87a" if r_avg >= 70 else "#ffe135" if r_avg >= 50 else "#c84b4b"
                    r_val_str = f"{r_avg:.1f}"
                except Exception:
                    r_col     = "#7aaac8"
                    r_val_str = str(region_avg)

                region_scores_html += (
                    f'<div style="display:flex;align-items:center;justify-content:space-between;'
                    f'padding:3px 0;border-bottom:1px solid #1a2a42;">'
                    f'<div style="display:flex;align-items:center;gap:4px;">'
                    f'<span style="color:{colour};font-size:10px;">&#9679;</span>'
                    f'<span style="font-size:9px;color:#7aaac8;font-family:monospace;">{region_name}</span>'
                    f'</div>'
                    f'<span style="font-size:11px;font-weight:700;color:{r_col};">{r_val_str}</span>'
                    f'</div>'
                )

        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=x_vals, y=plddt_vals,
            mode="lines",
            line=dict(color="#00d4ff", width=2),
            hovertemplate="Res %{x}<br>pLDDT: %{y:.1f}<extra></extra>",
            showlegend=False,
        ))
        fig.update_layout(
            margin=dict(l=8, r=8, t=8, b=8),
            height=190,
            autosize=True,
            paper_bgcolor="#070c1a",
            plot_bgcolor="#0a0f1e",
            shapes=shapes,
            annotations=annotations,
            yaxis=dict(
                title=dict(text="pLDDT", font=dict(size=12, color="#00d4ff"), standoff=6),
                side="right",
                range=[0, Y_MAX],
                tickfont=dict(size=11, color="#7aaac8"),
                tickvals=[0, 25, 50, 75, 100],
                showgrid=True,
                gridcolor="#1a2a42",
                zerolinecolor="#1a2a42",
            ),
            xaxis=dict(
                title=dict(text="Residue", font=dict(size=12, color="#7aaac8"), standoff=6),
                tickfont=dict(size=11, color="#7aaac8"),
                showgrid=False,
                zerolinecolor="#1a2a42",
                autorange=True,
            ),
        )

        plot_id  = f"plddt-{chain_name.lower()}"
        fig_json = fig.to_json()
        plot_html = (
            f'<div id="{plot_id}" style="height:190px;display:block;"></div>'
            f'<script>(function(){{'
            f'  var d={fig_json};'
            f'  delete d.layout.width;'
            f'  Plotly.newPlot("{plot_id}",d.data,d.layout,{{responsive:true,displayModeBar:false}});'
            f'}})();</script>'
        )

        sidebar_html = ""
        if region_scores_html:
            sidebar_html = f"""
<div style="flex:0 0 30%;border-left:1px solid #1a2a42;
            padding:8px 8px;overflow-y:auto;background:#080e1e;">
  <div style="font-size:8px;color:#3a5070;letter-spacing:0.14em;text-transform:uppercase;
              margin-bottom:6px;">Region pLDDT</div>
  {region_scores_html}
</div>"""

        cards.append(f"""
<div style="border:1px solid #1a2a42;border-radius:6px;margin-bottom:8px;background:#0a1020;overflow:hidden;">
  <div style="display:flex;align-items:center;justify-content:space-between;
              padding:5px 10px;background:rgba(255,255,255,0.03);border-bottom:1px solid #1a2a42;">
    <div style="display:flex;align-items:center;gap:6px;">
      <span style="color:{chain_colour};font-size:14px;">&#9679;</span>
      <span style="font-weight:700;color:#c8ddf0;font-size:11px;font-family:monospace;">{chain_name}</span>
      <span style="color:#3a5070;font-size:9px;">{len(plddt_vals)} residues</span>
    </div>
    <div style="display:flex;align-items:center;gap:4px;">
      <span style="font-size:9px;color:#4a6080;">avg</span>
      <span style="font-size:12px;font-weight:700;color:{avg_col};">{avg:.1f}</span>
    </div>
  </div>
  <div style="display:flex;">
    <div style="flex:0 0 70%;overflow:hidden;">{plot_html}</div>
    {sidebar_html}
  </div>
</div>""")

    if not cards:
        return '<div style="display:flex;align-items:center;justify-content:center;height:100%;color:#3a5070;font-style:italic;font-size:13px;">No pLDDT data available.</div>'

    return f'<div style="padding:7px 8px;">{"".join(cards)}</div>'


def build_components_html(struct_json, chain_info=None):
    """Build the Components sub-tab: Plot placeholder + Scores cards."""
    if struct_json is None:
        return '<div style="display:flex;align-items:center;justify-content:center;height:100%;color:#3a5070;font-style:italic;font-size:13px;">No JSON data available.</div>', "{}"

    components = struct_json.get("Components", {})
    scores     = components.get("Scores", {})

    # ── Plot placeholder ──────────────────────────────────────────────────────
    plot_section = """
<div style="border:1px solid #1a2a42;border-radius:6px;margin-bottom:10px;background:#0a1020;overflow:hidden;">
  <div style="padding:6px 10px;background:rgba(255,255,255,0.03);border-bottom:1px solid #1a2a42;">
    <span style="font-size:9px;font-weight:700;letter-spacing:0.16em;text-transform:uppercase;color:#4a6080;">Plot</span>
  </div>
  <div style="display:flex;align-items:center;justify-content:center;height:160px;color:#3a5070;font-style:italic;font-size:13px;">
    No information available
  </div>
</div>"""

    # ── Scores ────────────────────────────────────────────────────────────────
    cumulative = sum(v for v in scores.values() if isinstance(v, (int, float)))
    cum_col    = "#4bc87a" if cumulative == 0 else "#ffe135" if cumulative <= 2 else "#c84b4b"

    score_cards = ""
    for score_name, score_val in scores.items():
        if isinstance(score_val, (int, float)):
            if score_val == 0:
                symbol  = "&#10003;"   # ✓
                sym_col = "#4bc87a"
            else:
                symbol  = "&#10007;"   # ✗
                sym_col = "#c84b4b"
        else:
            symbol  = str(score_val)
            sym_col = "#7aaac8"

        square_placeholder = (
            '<div style="width:80px;height:80px;border-radius:4px;'
            'background:#0d1528;border:1px solid #1a2a42;flex-shrink:0;'
            'display:flex;align-items:center;justify-content:center;">'
            '<span style="color:#2a3a50;font-size:9px;">plot</span>'
            '</div>'
        )

        score_cards += f"""
<div style="border:1px solid #1a2a42;border-radius:6px;background:#0a1020;overflow:hidden;flex:1;min-width:0;">
  <div style="display:flex;flex-direction:column;align-items:center;padding:10px 8px;gap:6px;">
    {square_placeholder}
    <span style="font-size:8px;color:#4a6080;letter-spacing:0.12em;text-transform:uppercase;text-align:center;">{score_name}</span>
    <span style="font-size:28px;font-weight:700;color:{sym_col};">{symbol}</span>
  </div>
</div>"""

    scores_section = f"""
<div style="border:1px solid #1a2a42;border-radius:6px;margin-bottom:8px;background:#080e1e;overflow:hidden;">
  <div style="display:flex;align-items:center;justify-content:space-between;
              padding:6px 10px;background:rgba(255,255,255,0.03);border-bottom:1px solid #1a2a42;">
    <span style="font-size:9px;font-weight:700;letter-spacing:0.16em;text-transform:uppercase;color:#4a6080;">Scores</span>
    <div style="display:flex;align-items:center;gap:6px;">
      <span style="font-size:9px;color:#4a6080;">Cumulative</span>
      <span style="font-size:16px;font-weight:700;color:{cum_col};">{cumulative}</span>
    </div>
  </div>
  <div style="padding:8px;display:flex;flex-direction:row;gap:8px;">
    {score_cards if score_cards else '<p style="color:#3a5070;font-style:italic;font-size:12px;">No scores available.</p>'}
  </div>
</div>"""

    com_trace_map = chain_info.get("__com_trace_map__", {}) if chain_info else {}
    com_data      = components.get("COM", {})

    # ── COM pill box ─────────────────────────────────────────────────────────
    COM_GROUPS = {
        "Global": ["COM", "TCR", "MHC", "Pep"],
        "MHC":    ["MHCa", "MHCb", "MHCha", "MHChb"],
        "TCR":    ["TCRa", "TCRb", "TCRva", "TCRvb", "TCRca", "TCRcb"],
        "CDR":    ["CDR1a", "CDR2a", "CDR3a", "CDR1b", "CDR2b", "CDR3b"],
    }
    _REGION_PILL_COLOURS = [
        "#ff6b35", "#7dff6b", "#ff3de8", "#ffe135",
        "#b56bff", "#35ffc8", "#ff3535", "#00d4ff",
        "#ff9900", "#00ffcc", "#ff00aa", "#aaffaa",
    ]
    com_map_js = {}
    group_cards_html = ""
    colour_counter = 0

    for group_name, group_keys in COM_GROUPS.items():
        group_pills = ""
        for com_name in group_keys:
            if com_name not in com_trace_map:
                continue
            info   = com_trace_map[com_name]
            colour = info["colour"]
            com_map_js[com_name] = info["trace_idx"]
            xyz = com_data.get(com_name, [0, 0, 0])
            tip = f"{com_name} ({xyz[0]:.1f}, {xyz[2]:.1f}, {xyz[1]:.1f}) Å"
            group_pills += (
                f'<div class="com-pill" data-com="{com_name}" title="{tip}" '
                f'style="display:flex;align-items:center;gap:5px;'
                f'padding:4px 8px;margin-bottom:3px;border-radius:4px;cursor:pointer;'
                f'border:1px solid {colour};background:#0a1020;'
                f'font-family:monospace;font-size:10px;color:#c8ddf0;white-space:nowrap;">'
                f'<span style="color:{colour};font-size:11px;">&#9679;</span>'
                f'<span>{com_name}</span>'
                f'</div>\n'
            )
            colour_counter += 1
        if not group_pills:
            continue
        group_cards_html += f"""
<div style="flex-shrink:0;background:#0a1020;border:1px solid #1a2a42;
            border-radius:6px;overflow:hidden;min-width:100px;">
  <div style="display:flex;align-items:center;gap:5px;padding:5px 8px;
              background:rgba(255,255,255,0.03);border-bottom:1px solid #1a2a42;">
    <span style="font-weight:700;color:#c8ddf0;font-size:10px;
                 font-family:monospace;letter-spacing:0.06em;">{group_name}</span>
  </div>
  <div style="padding:6px 6px;display:flex;flex-direction:column;align-items:flex-start;">
    {group_pills}
  </div>
</div>"""

    plot_section = f"""
<div style="border:1px solid #1a2a42;border-radius:6px;margin-bottom:10px;background:#080e1e;overflow:hidden;">
  <div style="display:flex;align-items:center;justify-content:space-between;
              padding:6px 10px;background:rgba(255,255,255,0.03);border-bottom:1px solid #1a2a42;">
    <span style="font-size:9px;font-weight:700;letter-spacing:0.16em;text-transform:uppercase;color:#4a6080;">Centre of Mass</span>
    <span style="font-size:9px;color:#3a5070;font-style:italic;">Hover to show</span>
  </div>
  <div style="padding:6px 8px 8px;display:flex;flex-direction:row;flex-wrap:wrap;gap:6px;">
    {group_cards_html if group_cards_html else '<p style="color:#3a5070;font-style:italic;font-size:11px;">No COM data available.</p>'}
  </div>
</div>"""

    return f'<div style="padding:7px 8px;">{plot_section}{scores_section}</div>', json.dumps(com_map_js)


def build_structure_html(struct_json, chain_info=None):
    """
    Build the Structure tab content from the loaded JSON dict.
    Returns (html_str, trace_map_json_str).
    """
    if chain_info is None:
        chain_info = {}
    if struct_json is None:
        return (
            '<div style="display:flex;align-items:center;justify-content:center;'
            'height:100%;color:#3a5070;font-style:italic;font-size:13px;">'
            'No JSON file found for this structure.</div>',
            "{}"
        )

    regions_html, trace_map_json = build_regions_html(struct_json, chain_info)

    secondary    = struct_json.get("Secondary", {})
    cum_score    = secondary.get("Cumulative score", "—")
    score_colour = "#4bc87a" if isinstance(cum_score, (int, float)) and cum_score <= 3 else \
                   "#ffe135" if isinstance(cum_score, (int, float)) and cum_score <= 8 else "#c84b4b"

    cards = []

    for chain_name in _STRUCT_CHAIN_ORDER:
        chain_data = secondary.get(chain_name)
        if chain_data is None:
            continue

        annotation = chain_data.get("Annotation", [])
        sse        = chain_data.get("Structure", [])
        score_data = chain_data.get("Score", {})

        n_helix    = score_data.get("n_helix",   "—")
        exp_helix  = score_data.get("exp_helix", "—")
        n_sheet    = score_data.get("n_sheet",   "—")
        exp_sheet  = score_data.get("exp_sheet", "—")
        score      = score_data.get("score",     "—")

        chain_colour = _chain_col(_CHAIN_NAME_TO_IDX.get(chain_name, 0))

        region     = struct_json.get("Region", {}).get(chain_name, None)
        plot_html  = _sse_plot_html(annotation, sse, chain_name, region)

        def _stat_row(label, got, exp):
            match_col = "#4bc87a" if got == exp else "#c84b4b"
            return (
                f'<div style="display:flex;justify-content:space-between;'
                f'align-items:center;margin-bottom:4px;">'
                f'<span style="font-size:9px;color:#4a6080;">{label}</span>'
                f'<span style="font-size:10px;font-weight:700;color:{match_col};">'
                f'{got}<span style="color:#3a5070;font-weight:400;">/{exp}</span></span>'
                f'</div>'
            )

        score_col = "#4bc87a" if isinstance(score, (int, float)) and score == 0 else \
                    "#ffe135" if isinstance(score, (int, float)) and score <= 2 else "#c84b4b"

        stats_html = f"""
<div style="display:flex;flex-direction:column;justify-content:center;
            padding:8px 10px;gap:2px;">
  {_stat_row("Helices", n_helix, exp_helix)}
  {_stat_row("Sheets",  n_sheet, exp_sheet)}
  <div style="border-top:1px solid #1a2a42;margin:4px 0;"></div>
  <div style="display:flex;justify-content:space-between;align-items:center;">
    <span style="font-size:9px;color:#4a6080;">Score</span>
    <span style="font-size:14px;font-weight:700;color:{score_col};">{score}</span>
  </div>
</div>"""

        cards.append(f"""
<div style="border:1px solid #1a2a42;border-radius:6px;margin-bottom:8px;
            background:#0a1020;overflow:hidden;">
  <!-- Chain header -->
  <div style="display:flex;align-items:center;padding:5px 10px;
              background:rgba(255,255,255,0.03);border-bottom:1px solid #1a2a42;">
    <span style="color:{chain_colour};margin-right:6px;font-size:14px;">&#9679;</span>
    <span style="font-weight:700;color:#c8ddf0;font-size:11px;
                 letter-spacing:0.06em;font-family:monospace;">{chain_name}</span>
    <span style="color:#3a5070;font-size:9px;margin-left:8px;">{len(sse)} residues</span>
  </div>
  <!-- Plot 70% | Stats 30% -->
  <div style="display:flex;">
    <div style="flex:0 0 70%;overflow:hidden;">{plot_html}</div>
    <div style="flex:0 0 30%;border-left:1px solid #1a2a42;">{stats_html}</div>
  </div>
</div>""")

    if not cards:
        return (
            '<div style="display:flex;align-items:center;justify-content:center;'
            'height:100%;color:#3a5070;font-style:italic;font-size:13px;">'
            'No secondary structure data found.</div>',
            "{}"
        )

    sse_header = f"""
<div style="display:flex;align-items:center;justify-content:space-between;
            padding:8px 10px;border-bottom:1px solid #1a2a42;flex-shrink:0;
            background:#080e1e;">
  <span style="font-size:9px;letter-spacing:0.16em;text-transform:uppercase;
               color:#4a6080;">Secondary Structure</span>
  <div style="display:flex;align-items:center;gap:6px;">
    <span style="font-size:9px;color:#4a6080;">Cumulative Score</span>
    <span style="font-size:16px;font-weight:700;color:{score_colour};">{cum_score}</span>
  </div>
</div>"""

    sse_body = f'<div style="overflow-y:auto;flex:1;padding:7px 8px;">{"".join(cards)}</div>'
    sse_html = sse_header + sse_body

    return regions_html, sse_html, trace_map_json


# ── 8. Summary panel builder ───────────────────────────────────────────────────

# Placeholder booleans — replace each pair with real values later.
# Each entry: (bool_a, bool_b)
# Logic: both same → "UNKNOWN"; True/False mismatch → ✓ or ✗
STATUS_FLAGS = {
    "Folding Accuracy":    (None, None),
    "Structural Features": (None, None),
    "Feature Prediction":    (None, None),
    "Feature Prediction":  (None, None),
    "CNN Prediction":      (None, None),
}

def _status_display(bool_a, bool_b):
    """
    Return (symbol_html, colour) for a status box.
    Both None / both same value  → "UNKNOWN" in muted grey.
    True  + False (or vice-versa) → ✓ green  or  ✗ red
    based on which is True (True = positive result).
    """
    if bool_a is None and bool_b is None:
        return "UNKNOWN", "#4a6080"
    if bool_a == bool_b:
        return "UNKNOWN", "#4a6080"
    # one is True, one is False
    if bool_a is True or bool_b is True:
        return "&#10003;", "#4bc87a"   # green check
    return "&#10007;", "#c84b4b"        # red cross


def build_summary_html(struct_json, n_chains):
    if struct_json is None:
        return '<p style="color:#3a5070;font-style:italic;padding:12px;">No JSON file found.</p>'

    name      = str(struct_json.get("Name", "—")).strip()
    sequences = struct_json.get("Sequence", {})
    scoring   = struct_json.get("Scoring", {})
    prediction = struct_json.get("Predictions", {})

    # MHC class
    mhcb_val  = sequences.get("MHCb", {})
    mhcb_seq  = mhcb_val.get("Seq", "") if isinstance(mhcb_val, dict) else str(mhcb_val or "")
    mhc_class = "2" if mhcb_seq.strip() and mhcb_seq.strip().lower() not in ("nan","none","") else "1"

    # Peptide length
    pep_val = sequences.get("Peptide", {})
    pep_seq = pep_val.get("Seq", "") if isinstance(pep_val, dict) else str(pep_val or "")
    pep_len = str(len(pep_seq.strip())) if pep_seq.strip() else "—"

    def _info_box(label, value, value_colour="#00d4ff", font_size="20px"):
        return (
            f'<div style="flex:1;background:#0a1020;border:1px solid #1a2a42;'
            f'border-radius:6px;padding:10px 8px;text-align:center;min-width:0;">'
            f'<div style="font-size:8px;color:#4a6080;letter-spacing:0.10em;'
            f'text-transform:uppercase;margin-bottom:6px;white-space:nowrap;'
            f'overflow:hidden;text-overflow:ellipsis;">{label}</div>'
            f'<div style="font-size:{font_size};font-weight:700;color:{value_colour};">{value}</div>'
            f'</div>'
        )

    def _score_item(label, value, threshold, lower_is_better=True):
        """Row with label, numeric value, and check/x based on threshold."""
        try:
            v = float(value)
            if lower_is_better:
                ok = v <= threshold
            else:
                ok = v >= threshold
            sym     = "&#10003;" if ok else "&#10007;"
            sym_col = "#4bc87a"  if ok else "#c84b4b"
            val_col = "#4bc87a"  if ok else "#c84b4b"
            if isinstance(value, float):
                val_str = f"{v:.2f}" if abs(v) < 1000 else f"{v:.0f}"
            else:
                val_str = str(value)
        except Exception:
            sym = "—"; sym_col = "#3a5070"; val_col = "#7aaac8"; val_str = str(value)

        return (
            f'<div style="display:flex;align-items:center;justify-content:space-between;'
            f'padding:4px 0;border-bottom:1px solid #1a2a42;">'
            f'<span style="font-size:9px;color:#7aaac8;font-family:monospace;">{label}</span>'
            f'<div style="display:flex;align-items:center;gap:8px;">'
            f'<span style="font-size:11px;font-weight:700;color:{val_col};">{val_str}</span>'
            f'<span style="font-size:16px;font-weight:700;color:{sym_col};">{sym}</span>'
            f'</div>'
            f'</div>'
        )

    def _group_card(title, cards_html):
        return (
            f'<div style="margin-bottom:12px;">'
            f'<div style="font-size:8px;color:#3a5070;letter-spacing:0.16em;'
            f'text-transform:uppercase;margin-bottom:6px;">{title}</div>'
            f'<div style="display:flex;gap:8px;flex-wrap:wrap;">'
            f'{cards_html}'
            f'</div>'
            f'</div>'
        )

    def _section_card(title, items_html):
        return (
            f'<div style="flex:1;min-width:140px;border:1px solid #1a2a42;border-radius:6px;'
            f'background:#0a1020;overflow:hidden;">'
            f'<div style="padding:6px 10px;background:rgba(255,255,255,0.03);'
            f'border-bottom:1px solid #1a2a42;">'
            f'<span style="font-size:9px;font-weight:700;letter-spacing:0.16em;'
            f'text-transform:uppercase;color:#4a6080;">{title}</span>'
            f'</div>'
            f'<div style="padding:6px 10px;">{items_html}</div>'
            f'</div>'
        )

    # ── Structural Accuracy group ─────────────────────────────────────────────
    completeness = _section_card("Completeness",
        _score_item("Sequence",  scoring.get("Sequence",  "—"), 10) +
        _score_item("Consensus", scoring.get("Consensus", "—"), 10) +
        _score_item("Secondary", scoring.get("Secondary", "—"), 10) +
        _score_item("Region",    scoring.get("Region",    "—"), 0)
    )
    fidelity = _section_card("Structural Fidelity",
        _score_item("Peptide Position", scoring.get("Peptide position",  "—"), 0) +
        _score_item("pMHC Structure",   scoring.get("pMHC structure",    "—"), 0) +
        _score_item("MHC Orientation",  scoring.get("MHC orientation",   "—"), 0)
    )
    pred_accuracy = _section_card("Predicted Accuracy",
        _score_item("Total pLDDT",    scoring.get("Total pLDDT",    "—"), 70, lower_is_better=False) +
        _score_item("Variable pLDDT", scoring.get("Variable pLDDT", "—"), 70, lower_is_better=False)
    )
    struct_accuracy = _group_card("Structural Accuracy",
                                  completeness + fidelity + pred_accuracy)

    # ── Interaction Prediction group ──────────────────────────────────────────
    pred_items = ""
    for pred_name, pred_data in prediction.items():
        if not isinstance(pred_data, dict):
            continue
        prob   = pred_data.get("Probability", None)
        result = pred_data.get("Interact",     None)
        try:
            p_f     = float(prob)
            p_str   = f"{p_f:.3f}"
            p_col   = "#4bc87a" if p_f >= 0.5 else "#c84b4b"
        except Exception:
            p_str = str(prob) if prob is not None else "—"
            p_col = "#7aaac8"
        sym     = ("&#10003;" if result is True  else
                   "&#10007;" if result is False else "—")
        sym_col = ("#4bc87a"  if result is True  else
                   "#c84b4b"  if result is False else "#3a5070")
        pred_items += (
            f'<div style="display:flex;align-items:center;justify-content:space-between;'
            f'padding:4px 0;border-bottom:1px solid #1a2a42;">'
            f'<span style="font-size:9px;color:#7aaac8;font-family:monospace;">{pred_name}</span>'
            f'<div style="display:flex;align-items:center;gap:8px;">'
            f'<span style="font-size:11px;font-weight:700;color:{p_col};">{p_str}</span>'
            f'<span style="font-size:16px;font-weight:700;color:{sym_col};">{sym}</span>'
            f'</div>'
            f'</div>'
        )

    interaction = _group_card("Interaction Prediction",
        _section_card("Predictions", pred_items) if pred_items
        else _section_card("Predictions",
            '<p style="color:#3a5070;font-style:italic;font-size:11px;">No predictions available.</p>')
    )

    return f"""
<div style="padding:12px 10px;overflow-y:auto;">
  <!-- Name box -->
  <div style="background:#0a1020;border:1px solid #1a2a42;border-radius:6px;
              padding:12px 14px;margin-bottom:12px;">
    <div style="font-size:8px;color:#4a6080;letter-spacing:0.14em;
                text-transform:uppercase;margin-bottom:4px;">Structure</div>
    <div style="font-size:15px;font-weight:700;color:#c8ddf0;
                font-family:'Space Mono',monospace;word-break:break-all;">{name}</div>
  </div>

  <!-- Info boxes -->
  <div style="display:flex;gap:8px;margin-bottom:12px;">
    {_info_box("MHC Class", mhc_class)}
    {_info_box("# Chains", str(n_chains))}
    {_info_box("Peptide Length", pep_len)}
  </div>

  {struct_accuracy}
  {interaction}
</div>"""

# ── 8. PDB file list ───────────────────────────────────────────────────────────
def get_pdb_files():
    """Scan driveDir/Results/<stem>/Structure/<stem>.pdb for all structures."""
    pattern = os.path.join(RESULTS_DIR, "*", "Structure", "*.pdb")
    return sorted(glob.glob(pattern))

# ── 9. Full-page HTML renderer ─────────────────────────────────────────────────
def build_prediction_html(struct_json):
    """Build the Prediction tab: one card per prediction, displayed horizontally."""
    if struct_json is None:
        return _placeholder_panel()

    predictions = struct_json.get("Predictions", {})
    if not predictions:
        return _placeholder_panel()

    cards = ""
    for pred_name, pred_data in predictions.items():
        if not isinstance(pred_data, dict):
            continue

        prob   = pred_data.get("Probability", None)
        result = pred_data.get("Interact",     None)

        # Probability
        if prob is not None:
            try:
                prob_f   = float(prob)
                prob_col = "#4bc87a" if prob_f >= 0.5 else "#c84b4b"
                prob_str = f"{prob_f:.3f}"
            except Exception:
                prob_col = "#7aaac8"
                prob_str = str(prob)
        else:
            prob_col = "#3a5070"
            prob_str = "—"

        # Result boolean → check/x
        if result is True:
            sym     = "&#10003;"
            sym_col = "#4bc87a"
        elif result is False:
            sym     = "&#10007;"
            sym_col = "#c84b4b"
        else:
            sym     = "—"
            sym_col = "#3a5070"

        cards += f"""
<div style="flex:1;min-width:140px;border:1px solid #1a2a42;border-radius:6px;
            background:#0a1020;overflow:hidden;">
  <div style="padding:6px 10px;background:rgba(255,255,255,0.03);
              border-bottom:1px solid #1a2a42;text-align:center;">
    <span style="font-size:10px;font-weight:700;color:#c8ddf0;
                 font-family:monospace;letter-spacing:0.06em;">{pred_name}</span>
  </div>
  <div style="display:flex;flex-direction:column;align-items:center;
              gap:10px;padding:14px 10px;">
    <div style="display:flex;flex-direction:column;align-items:center;gap:2px;">
      <span style="font-size:8px;color:#4a6080;letter-spacing:0.12em;
                   text-transform:uppercase;">Probability</span>
      <span style="font-size:22px;font-weight:700;color:{prob_col};">{prob_str}</span>
    </div>
    <div style="display:flex;flex-direction:column;align-items:center;gap:2px;">
      <span style="font-size:8px;color:#4a6080;letter-spacing:0.12em;
                   text-transform:uppercase;">Result</span>
      <span style="font-size:28px;font-weight:700;color:{sym_col};">{sym}</span>
    </div>
  </div>
</div>"""

    if not cards:
        return _placeholder_panel()

    return f'<div style="padding:10px;display:flex;flex-wrap:wrap;gap:10px;">{cards}</div>'


def render_page(selected_path, pdb_files, color_scheme="chain",
                show_points=False, show_lines=True, show_ss=False):
    stem = os.path.splitext(os.path.basename(selected_path or ""))[0]

    # Load JSON first so it can be passed to build_figure_dict
    struct_json = load_structure_json(stem)

    # Build figure
    if selected_path and os.path.exists(selected_path):
        fig_json, n_ca, n_res, n_chains, chain_info = build_figure_dict(
            selected_path, color_scheme=color_scheme,
            show_points=show_points, show_lines=show_lines,
            show_ss=show_ss, struct_json=struct_json,
        )
    else:
        fig = go.Figure()
        fig.update_layout(paper_bgcolor="#070c1a", scene=dict(bgcolor="#080d1c"),
                          margin=dict(l=0,r=0,t=0,b=0))
        fig_json   = fig.to_json()
        n_ca       = n_res = n_chains = 0
        chain_info = {}

    # Panel HTML — all driven from struct_json, no CSV needed
    try:
        seq_html = build_sequence_html(struct_json)
    except Exception as e:
        seq_html = _placeholder_panel(f"Sequence error: {e}")

    try:
        consensus_html = build_consensus_html(struct_json)
    except Exception as e:
        consensus_html = _placeholder_panel(f"Consensus error: {e}")

    try:
        sum_html = build_summary_html(struct_json, n_chains)
    except Exception as e:
        sum_html = _placeholder_panel(f"Summary error: {e}")

    try:
        regions_html, sse_html, trace_map_json = build_structure_html(struct_json, chain_info)
    except Exception as e:
        regions_html = sse_html = _placeholder_panel(f"Structure error: {e}")
        trace_map_json = "{}"

    try:
        plddt_html = build_plddt_html(chain_info, struct_json)
    except Exception as e:
        plddt_html = _placeholder_panel(f"pLDDT error: {e}")

    try:
        components_html, com_map_json = build_components_html(struct_json, chain_info)
    except Exception as e:
        components_html = _placeholder_panel(f"Components error: {e}")
        com_map_json = "{}"

    try:
        prediction_html = build_prediction_html(struct_json)
    except Exception as e:
        prediction_html = _placeholder_panel(f"Prediction error: {e}")

    geom_trace_count = chain_info.get("__geom_trace_count__", 0)

    _na_panel = (
        '<div style="display:flex;align-items:center;justify-content:center;'
        'height:100%;color:#3a5070;font-style:italic;font-size:13px;">'
        'No information available</div>'
    )

    # Dropdown options
    options_html = ""
    for f in pdb_files:
        label    = os.path.basename(f)
        selected = 'selected' if f == selected_path else ''
        encoded  = f.replace("&", "%26").replace('"', "%22")
        options_html += f'<option value="{encoded}" {selected}>{label}</option>\n'
    if not pdb_files:
        options_html = '<option value="">⚠ No PDB files found</option>'

    # Stats bar values
    stats_html = f"""
<div style="border-top:1px solid #1b2d4f;padding:6px 10px;display:flex;gap:18px;
            flex-shrink:0;background:#0d1528;">
  <div style="display:flex;flex-direction:column;gap:1px;">
    <span style="font-size:8px;color:#4a6080;letter-spacing:0.1em;text-transform:uppercase;">Atoms</span>
    <span style="font-size:13px;color:#00d4ff;font-weight:700;">{n_ca}</span>
  </div>
  <div style="display:flex;flex-direction:column;gap:1px;">
    <span style="font-size:8px;color:#4a6080;letter-spacing:0.1em;text-transform:uppercase;">Residues</span>
    <span style="font-size:13px;color:#00d4ff;font-weight:700;">{n_res}</span>
  </div>
  <div style="display:flex;flex-direction:column;gap:1px;">
    <span style="font-size:8px;color:#4a6080;letter-spacing:0.1em;text-transform:uppercase;">Chains</span>
    <span style="font-size:13px;color:#00d4ff;font-weight:700;">{n_chains}</span>
  </div>
</div>"""

    # Right panel toolbar options
    scheme_options = "".join(
        f'<option value="{s}" {"selected" if s == color_scheme else ""}>'
        f'{"pLDDT" if s == "plddt" else s.replace("_"," ").title()}</option>'
        for s in COLOR_SCHEMES
    )
    pts_checked   = "checked" if show_points else ""
    lines_checked = "checked" if show_lines  else ""
    ss_checked    = "checked" if show_ss     else ""

    # Encode current mol path for use in JS navigation
    mol_encoded = selected_path.replace("&", "%26").replace('"', "%22") if selected_path else ""

    return f"""<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>MolViewer</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=Syne:wght@400;700;800&display=swap" rel="stylesheet">
<style>
  *, *::before, *::after {{ box-sizing: border-box; margin: 0; padding: 0; }}
  html, body {{ height: 100%; overflow: hidden;
                font-family: 'Space Mono', monospace;
                background: #070c1a; color: #c8ddf0; }}

  /* ── header ── */
  #hdr {{ display:flex; align-items:center; gap:10px; padding:8px 14px;
          background:#0d1528; border-bottom:1px solid #1b2d4f; flex-shrink:0; }}
  .logo {{ font-family:'Syne',sans-serif; font-weight:800; font-size:17px;
           color:#00d4ff; letter-spacing:-0.02em; white-space:nowrap; }}
  .logo b {{ color:#ff6b35; }}
  .divider {{ width:1px; height:22px; background:#1b2d4f; }}
  .hdr-lbl {{ font-size:9px; color:#4a6080; letter-spacing:0.14em;
              text-transform:uppercase; white-space:nowrap; }}
  #mol-select {{
    background:#070c1a; color:#00d4ff; border:1px solid #1b2d4f;
    border-radius:4px; font-family:'Space Mono',monospace; font-size:12px;
    padding:4px 8px; min-width:300px; outline:none; cursor:pointer;
  }}
  #refresh-btn {{
    margin-left:auto; background:transparent; border:1px solid #1b2d4f;
    border-radius:4px; color:#4a6080; font-family:'Space Mono',monospace;
    font-size:10px; letter-spacing:0.1em; padding:4px 10px; cursor:pointer;
    transition:color 0.15s, border-color 0.15s;
  }}
  #refresh-btn:hover {{ color:#00d4ff; border-color:#00d4ff; }}

  /* ── shell ── */
  #shell {{ display:flex; height:calc(100vh - 44px); overflow:hidden; }}

  /* ── left panel 60% ── */
  #lpanel {{ width:60%; flex-shrink:0; background:#0d1528;
             border-right:1px solid #1b2d4f;
             display:flex; flex-direction:column; overflow:hidden; }}

  /* ── tab bar ── */
  #tab-bar {{ display:flex; flex-shrink:0; border-bottom:1px solid #1b2d4f;
              background:#080e1e; }}
  .tab-btn {{
    flex:1; padding:8px 4px; font-family:'Space Mono',monospace;
    font-size:10px; letter-spacing:0.08em; text-transform:uppercase;
    color:#4a6080; background:transparent; border:none; border-bottom:2px solid transparent;
    cursor:pointer; transition:color 0.15s, border-color 0.15s;
    white-space:nowrap;
  }}
  .tab-btn:hover  {{ color:#7aaac8; }}
  .tab-btn.active {{ color:#00d4ff; border-bottom-color:#00d4ff; }}

  /* ── tab content ── */
  .tab-panel {{ display:none; flex:1; overflow:hidden; flex-direction:column; }}
  .tab-panel.active {{ display:flex; }}
  #tab-summary  {{ overflow-y:auto; }}
  #tab-structure {{ overflow:hidden; }}

  /* ── sub-tabs (used in Structure, Sequence, Physics, Prediction) ── */
  .sub-bar {{
    display:flex; flex-shrink:0; background:#060b18;
    border-bottom:1px solid #1b2d4f;
  }}
  .sub-btn {{
    flex:1; padding:6px 4px;
    font-family:'Space Mono',monospace; font-size:9px;
    letter-spacing:0.08em; text-transform:uppercase;
    color:#3a5070; background:transparent; border:none;
    border-bottom:2px solid transparent;
    cursor:pointer; transition:color 0.15s, border-color 0.15s;
    white-space:nowrap;
  }}
  .sub-btn:hover  {{ color:#7aaac8; }}
  .sub-btn.active {{ color:#00d4ff; border-bottom-color:#00d4ff; }}
  .sub-panel {{ display:none; flex:1; overflow-y:auto; flex-direction:column; }}
  .sub-panel.active {{ display:flex; }}

  /* ── right 40% ── */
  #rpanel {{ flex:1; overflow:hidden; display:flex; flex-direction:column; background:#070c1a; }}
  #mol-graph {{ flex:1; min-height:0; }}

  /* ── right toolbar ── */
  #rtoolbar {{
    display:flex; align-items:center; gap:8px; padding:5px 10px;
    background:#0d1528; border-bottom:1px solid #1b2d4f; flex-shrink:0;
  }}
  .rt-lbl {{ font-size:9px; color:#4a6080; letter-spacing:0.12em;
             text-transform:uppercase; white-space:nowrap; }}
  .rt-select {{
    background:#070c1a; color:#00d4ff; border:1px solid #1b2d4f;
    border-radius:4px; font-family:'Space Mono',monospace; font-size:11px;
    padding:3px 6px; outline:none; cursor:pointer;
  }}
  .rt-toggle {{
    display:flex; align-items:center; gap:5px; cursor:pointer;
    font-size:10px; color:#7aaac8; user-select:none;
  }}
  .rt-toggle input {{ accent-color:#00d4ff; cursor:pointer; }}
</style>
</head>
<body>

<!-- Header -->
<div id="hdr">
  <span class="logo">Mol<b>Viewer</b></span>
  <span class="divider"></span>
  <span class="hdr-lbl">Structure</span>
  <select id="mol-select" onchange="navWithParams({{mol: encodeURIComponent(this.value)}})">
    {options_html}
  </select>
  <button id="refresh-btn" onclick="navWithParams({{}})">⟳ Refresh</button>
</div>

<!-- Body -->
<div id="shell">

  <!-- Left panel -->
  <div id="lpanel">

    <!-- Tab bar -->
    <div id="tab-bar">
      <button class="tab-btn" data-tab="summary"    onclick="switchTab('summary',   this)">Summary</button>
      <button class="tab-btn" data-tab="sequence"   onclick="switchTab('sequence',  this)">Sequence</button>
      <button class="tab-btn" data-tab="structure"  onclick="switchTab('structure', this)">Structure</button>
      <button class="tab-btn" data-tab="physics"    onclick="switchTab('physics',   this)">Features</button>
      <button class="tab-btn" data-tab="prediction" onclick="switchTab('prediction',this)">Prediction</button>
    </div>

    <!-- Summary tab (no sub-tabs) -->
    <div id="tab-summary" class="tab-panel active">
      {sum_html}
    </div>

    <!-- Sequence tab -->
    <div id="tab-sequence" class="tab-panel">
      <div class="sub-bar" id="seq-sub-bar">
        <button class="sub-btn active" data-sub="seq-sequence"  onclick="switchSubTab('seq-sequence',  this, 'seq-sub-bar')">Sequence</button>
        <button class="sub-btn"        data-sub="seq-consensus" onclick="switchSubTab('seq-consensus', this, 'seq-sub-bar')">Consensus</button>
      </div>
      <div id="seq-sequence"  class="sub-panel active" style="overflow-y:auto;padding:7px 8px;">
        {seq_html}
      </div>
      <div id="seq-consensus" class="sub-panel" style="overflow-y:auto;padding:7px 8px;">
        {consensus_html}
      </div>
    </div>

    <!-- Structure tab -->
    <div id="tab-structure" class="tab-panel" style="overflow:hidden;">
      <div class="sub-bar" id="struct-sub-bar">
        <button class="sub-btn active" data-sub="struct-regions"    onclick="switchSubTab('struct-regions',    this, 'struct-sub-bar')">Regions</button>
        <button class="sub-btn"        data-sub="struct-secondary"  onclick="switchSubTab('struct-secondary',  this, 'struct-sub-bar')">Secondary</button>
        <button class="sub-btn"        data-sub="struct-plddt"      onclick="switchSubTab('struct-plddt',      this, 'struct-sub-bar')">pLDDT</button>
        <button class="sub-btn"        data-sub="struct-components" onclick="switchSubTab('struct-components', this, 'struct-sub-bar')">Components</button>
      </div>
      <div id="struct-regions"    class="sub-panel active">{regions_html}</div>
      <div id="struct-secondary"  class="sub-panel" style="flex-direction:column;">{sse_html}</div>
      <div id="struct-plddt"      class="sub-panel" style="overflow-y:auto;padding:7px 8px;">{plddt_html}</div>
      <div id="struct-components" class="sub-panel" style="overflow-y:auto;">{components_html}</div>
    </div>

    <!-- Physics tab -->
    <div id="tab-physics" class="tab-panel" style="overflow:hidden;">
      <div class="sub-bar" id="phys-sub-bar">
        <button class="sub-btn active" data-sub="phys-main" onclick="switchSubTab('phys-main', this, 'phys-sub-bar')">Features</button>
      </div>
      <div id="phys-main" class="sub-panel active">
        <div style="display:flex;align-items:center;justify-content:center;height:100%;color:#3a5070;font-style:italic;font-size:13px;">No information available</div>
      </div>
    </div>

    <!-- Prediction tab -->
    <div id="tab-prediction" class="tab-panel" style="overflow:hidden;">
      <div class="sub-bar" id="pred-sub-bar">
        <button class="sub-btn active" data-sub="pred-main" onclick="switchSubTab('pred-main', this, 'pred-sub-bar')">Prediction</button>
      </div>
      <div id="pred-main" class="sub-panel active" style="overflow-y:auto;">
        {prediction_html}
      </div>
    </div>

    <!-- Stats bar (always visible) -->
    {stats_html}

  </div><!-- /lpanel -->

  <!-- Right: toolbar + 3-D plot -->
  <div id="rpanel">

    <!-- Toolbar -->
    <div id="rtoolbar">
      <span class="rt-lbl">Colour</span>
      <select class="rt-select" id="scheme-select">
        {scheme_options}
      </select>
      <span class="divider"></span>
      <label class="rt-toggle">
        <input type="checkbox" id="toggle-points" {pts_checked}>
        Points
      </label>
      <label class="rt-toggle">
        <input type="checkbox" id="toggle-lines" {lines_checked}>
        Lines
      </label>
      <label class="rt-toggle">
        <input type="checkbox" id="toggle-ss" {ss_checked}>
        SS
      </label>
    </div>

    <div id="mol-graph"></div>
  </div>

</div><!-- /shell -->

<script>
  function toggleSection(bodyId, header) {{
    const body  = document.getElementById(bodyId);
    const arrow = header.querySelector('.collapse-arrow');
    if (!body) return;
    const collapsed = body.style.display === 'none';
    const defaultDisplay = bodyId === 'regions-body' ? 'flex' : '';
    body.style.display  = collapsed ? defaultDisplay : 'none';
    if (arrow) arrow.style.transform = collapsed ? '' : 'rotate(180deg)';
  }}

  function switchSubTab(subId, btn, barId) {{
    const bar = document.getElementById(barId);
    if (!bar) return;
    // Deactivate all sub-panels and buttons in this bar's parent
    bar.querySelectorAll('.sub-btn').forEach(b => b.classList.remove('active'));
    // Find the parent tab-panel and deactivate its sub-panels
    const parent = bar.closest('.tab-panel');
    if (parent) parent.querySelectorAll('.sub-panel').forEach(p => p.classList.remove('active'));
    btn.classList.add('active');
    const panel = document.getElementById(subId);
    if (panel) panel.classList.add('active');
    // Persist in URL
    const u = new URL(window.location);
    u.searchParams.set('subtab', subId);
    window.history.replaceState({{}}, '', u);
  }}

  // Restore active tab from URL param
  const urlParams = new URLSearchParams(window.location.search);
  const activeTab = urlParams.get('tab') || 'summary';
  document.querySelectorAll('.tab-panel').forEach(p => p.classList.remove('active'));
  document.querySelectorAll('.tab-btn').forEach(b => b.classList.remove('active'));
  document.getElementById('tab-' + activeTab).classList.add('active');
  document.querySelector(`.tab-btn[data-tab="${{activeTab}}"]`).classList.add('active');

  // Restore active sub-tab from URL param
  const activeSubTab = urlParams.get('subtab');
  if (activeSubTab) {{
    const subPanel = document.getElementById(activeSubTab);
    if (subPanel) {{
      const parent = subPanel.closest('.tab-panel');
      if (parent) parent.querySelectorAll('.sub-panel').forEach(p => p.classList.remove('active'));
      subPanel.classList.add('active');
      // Find and activate the matching button
      const btn = document.querySelector(`.sub-btn[data-sub="${{activeSubTab}}"]`);
      if (btn) {{
        const bar = btn.closest('.sub-bar');
        if (bar) bar.querySelectorAll('.sub-btn').forEach(b => b.classList.remove('active'));
        btn.classList.add('active');
      }}
    }}
  }}

  function switchTab(name, btn) {{
    document.querySelectorAll('.tab-panel').forEach(p => p.classList.remove('active'));
    document.querySelectorAll('.tab-btn').forEach(b => b.classList.remove('active'));
    document.getElementById('tab-' + name).classList.add('active');
    btn.classList.add('active');
    // Update URL without reload so tab persists on next refresh
    const u = new URL(window.location);
    u.searchParams.set('tab', name);
    window.history.replaceState({{}}, '', u);
  }}

  // Build a URL preserving all current view params
  function navWithParams(overrides) {{
    const u = new URL(window.location);
    const mol    = overrides.mol    ?? u.searchParams.get('mol')    ?? encodeURIComponent("{mol_encoded}");
    const scheme = overrides.scheme ?? document.getElementById('scheme-select').value;
    const pts    = overrides.pts    ?? (document.getElementById('toggle-points').checked ? '1' : '0');
    const lines  = overrides.lines  ?? (document.getElementById('toggle-lines').checked  ? '1' : '0');
    const ss     = overrides.ss     ?? (document.getElementById('toggle-ss').checked     ? '1' : '0');
    const tab    = overrides.tab    ?? u.searchParams.get('tab')    ?? 'summary';
    const subtab = overrides.subtab ?? u.searchParams.get('subtab') ?? '';
    let url = `/?mol=${{mol}}&scheme=${{scheme}}&pts=${{pts}}&lines=${{lines}}&ss=${{ss}}&tab=${{tab}}`;
    if (subtab) url += `&subtab=${{subtab}}`;
    window.location.href = url;
  }}


  // ── Region hover highlighting ──────────────────────────────────────────────
  const TRACE_MAP   = {trace_map_json};
  const tooltip     = document.getElementById('region-tooltip');
  const graphDiv    = document.getElementById('mol-graph');

  const geomCount   = Object.values(TRACE_MAP)[0]?.geom_trace_count ?? 0;
  const geomIdx     = Array.from({{length: geomCount}}, (_, i) => i);
  const allHlIdx    = Object.values(TRACE_MAP).map(r => r.highlight_trace);

  // Keys whose highlight traces are currently visible
  const visibleKeys = new Set();
  // Keys that are click-locked
  const lockedKeys  = new Set();

  function _applyState() {{
    // Compute which highlight indices should be visible
    const showIdx = Array.from(visibleKeys).map(k => TRACE_MAP[k].highlight_trace);
    const hideIdx = allHlIdx.filter(i => !showIdx.includes(i));

    const shouldDim = visibleKeys.size > 0;

    if (shouldDim) {{
      Plotly.restyle(graphDiv, {{'opacity': 0.1}}, geomIdx);
    }} else {{
      Plotly.restyle(graphDiv, {{'opacity': 1}}, geomIdx);
    }}
    if (showIdx.length > 0) Plotly.restyle(graphDiv, {{'opacity': 1}}, showIdx);
    if (hideIdx.length > 0) Plotly.restyle(graphDiv, {{'opacity': 0}}, hideIdx);
  }}

  document.querySelectorAll('.region-pill').forEach(pill => {{
    const key = pill.dataset.key;

    pill.addEventListener('click', () => {{
      if (lockedKeys.has(key)) {{
        lockedKeys.delete(key);
        pill.style.boxShadow = '';
        // Only remove from visible if we're not hovering it
        // (mouseleave will handle that); remove directly since click fires after enter
        visibleKeys.delete(key);
      }} else {{
        lockedKeys.add(key);
        pill.style.boxShadow = '0 0 0 2px #00d4ff';
        visibleKeys.add(key);
      }}
      _applyState();
    }});

    pill.addEventListener('mouseenter', (e) => {{
      visibleKeys.add(key);
      _applyState();
      const tip = pill.dataset.tip;
      const seq = pill.dataset.seq;
      tooltip.innerHTML =
        `<div style="font-weight:700;color:#00d4ff;margin-bottom:4px;">${{tip}}</div>` +
        (seq ? `<div style="color:#7aaac8;letter-spacing:0.04em;word-break:break-all;">${{seq}}</div>` : '');
      tooltip.style.display = 'block';
    }});

    pill.addEventListener('mousemove', (e) => {{
      tooltip.style.left = (e.clientX + 14) + 'px';
      tooltip.style.top  = (e.clientY - 10) + 'px';
    }});

    pill.addEventListener('mouseleave', () => {{
      tooltip.style.display = 'none';
      if (!lockedKeys.has(key)) {{
        visibleKeys.delete(key);
        _applyState();
      }}
    }});
  }});
  document.getElementById('scheme-select').addEventListener('change', () => navWithParams({{}}));
  document.getElementById('toggle-points').addEventListener('change', () => navWithParams({{}}));
  document.getElementById('toggle-lines').addEventListener('change',  () => navWithParams({{}}));
  document.getElementById('toggle-ss').addEventListener('change',     () => navWithParams({{}}));

  const figData = {fig_json};
  Plotly.newPlot('mol-graph', figData.data, figData.layout, {{
    responsive: true,
    displaylogo: false,
    scrollZoom: true,
    modeBarButtonsToRemove: ['toImage'],
  }});
</script>
</body>
</html>"""

# ── 10. Flask app ──────────────────────────────────────────────────────────────
def _build_flask_app():
    app = Flask(__name__)

    import traceback

    # Temporarily show full tracebacks in the browser for debugging
    @app.errorhandler(Exception)
    def handle_exception(e):
        tb = traceback.format_exc()
        return Response(
            f"<pre style='color:red;background:#111;padding:20px;font-size:12px;'>"
            f"{tb}</pre>",
            status=500, mimetype="text/html"
        )

    logging.getLogger("werkzeug").setLevel(logging.WARNING)
    logging.getLogger("root").setLevel(logging.CRITICAL)

    @app.route("/")
    def index():
        pdb_files    = get_pdb_files()
        selected     = request.args.get("mol",    pdb_files[0] if pdb_files else "")
        color_scheme = request.args.get("scheme", "chain")
        show_points  = request.args.get("pts",    "0") == "1"
        show_lines   = request.args.get("lines",  "1") == "1"
        show_ss      = request.args.get("ss",     "0") == "1"

        if selected not in pdb_files:
            selected = pdb_files[0] if pdb_files else ""
        if color_scheme not in COLOR_SCHEMES:
            color_scheme = "chain"

        html = render_page(selected, pdb_files,
                           color_scheme=color_scheme,
                           show_points=show_points,
                           show_lines=show_lines,
                           show_ss=show_ss)
        return Response(html, mimetype="text/html")

    return app

# ── 11. Free-port finder ───────────────────────────────────────────────────────
def _free_port(preferred=8050):
    for port in range(preferred, preferred + 20):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex(("localhost", port)) != 0:
                return port
    return preferred

# ── 12. MolViewer class ────────────────────────────────────────────────────────
class MolViewer:
    """
    Starts a plain Flask server in a background daemon thread and displays
    the app via Colab's port proxy. No WebSockets, no Dash callbacks —
    the dropdown does a normal page navigation, Flask re-renders the full
    HTML, Plotly.js renders the figure from JSON baked into the page.

    Usage:
        viewer = MolViewer()   # starts server, cell returns immediately
        viewer.show()          # display iframe in current cell
        viewer.show()          # call again in any later cell
    """

    def __init__(self, port: int = 8050, height: int = IFRAME_H):
        self.port   = _free_port(port)
        self.height = height
        self._app   = _build_flask_app()
        self._start()

    def _start(self):
        def _run():
            self._app.run(host="0.0.0.0", port=self.port,
                          debug=False, use_reloader=False, threaded=True)

        t = threading.Thread(target=_run, daemon=True)
        t.start()

        # Wait until the port is accepting connections
        for _ in range(40):
            time.sleep(0.15)
            with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
                if s.connect_ex(("localhost", self.port)) == 0:
                    break

        print(f"MolViewer running on port {self.port}")

    def show(self, height: int = None):
        """Display the viewer iframe in the current Colab cell."""
        from google.colab import output
        output.serve_kernel_port_as_iframe(self.port, height=str(height or self.height))

In [12]:
#@title Visualizer
# from google.colab import output
# @markdown Run the Visualizer?
Run = True #@param {type:"boolean"}
# @markdown Choose the visualizer mode:
window_height = 720 #@param

#Mode currently does not work
#****Work in progress*******
Mode = "inline-opens below" #@param ["inline-opens below","external-new window follow link","tab-automatically new tab"]
if Mode == "external-new window follow link":
  Mode = "external"
elif Mode == "tab-automatically new tab":
  Mode = "tab"
elif Mode == "inline-opens below":
  Mode = "inline"

if Run:
#Make a window viewer so that people can see the models as they are generated and have all results in one place
#```` work in progress ``````#
  # app.server.run(jupyter_mode=Mode, jupyter_height=window_height, port=8050)

  # app.run_server(jupyter_mode=Mode, jupyter_height=window_height,port=8050, debug=False)
  # output.serve_kernel_port_as_window(8050)

  # viewer = MolViewer()
  # viewer.show()
  # from mol_viewer import MolViewer
      viewer = MolViewer()      # server starts in background
      viewer.show()             # renders iframe in this cell

 * Serving Flask app '__main__'
 * Debug mode: off
MolViewer running on port 8054


AttributeError: type object 'TermColors' has no attribute 'Green'